# Procesamiento de la Tesis: Modelo de Control por Razón de Bowen (BREB)

El presente *notebook* presenta el procesamiento necesario para transformar los **datos meteorológicos crudos**, calcular la **radiación neta ($R_n$)** y formular el **balance de energía en la superficie** mediante el método de la **Razón de Bowen ($\beta$)**. Para ello se recomienda primero ejecutar el archivo de preprocesamiento para obtener un archivo con los datos unificados.

> ⚠️ **Nota metodológica:** Este procesamiento constituye el **patrón de control experimental (benchmark)** contra el cual se contrastarán y evaluarán los diferentes modelos empíricos y teóricos de evapotranspiración real ($ET_a$) en ambiente protegido.
> Para el uso simplificado del notebook, se recomienda editar unicamente las celdas con el simbolo &#x1F527;. EN cambio las celdas con el simbolo &#x1F9E0; son revisiones teóricas de lo que se está realizando.

---

## 1. Entorno de Cómputo y Reproducibilidad

<div style="background-color: #f8fafc; border-left: 4px solid #3b82f6; padding: 14px 18px; border-radius: 6px; margin: 15px 0; color: #1e293b;">
    <p style="font-weight: bold; margin-top: 0; margin-bottom: 8px; color: #1d4ed8;">
        💻 Entorno de Ejecución
    </p>
    <ul style="margin: 0; padding-left: 20px; line-height: 1.6;">
        <li><b>Sistema Operativo:</b> Ubuntu 26.04.1 LTS (Kernel: <code>7.0.0-38-generic</code>)</li>
        <li><b>Arquitectura:</b> <code>x86_64</code></li>
        <li><b>Gestor de Entornos:</b> Mamba 2.9.0</li>
        <li><b>Entorno de Python:</b> <code>/home/alejandro/miniforge3/envs/tesis</code></li>
        <li><b>Canales (Channels):</b> <code>conda-forge</code></li>
        <li><b>Jupyter:</b> 8.7.0 | <b>Python:</b> 3.14.8</li>
    </ul>
</div>

---

## 2. Arquitectura de Sensores e Instrumentación
 
**Dispositivo principal:** Registrador de datos METER ZL6 (`z6-13472`)  
**Dispositivo autónomo:** Registrador de datos psicrométrico EL-USB-2-LCD / EL-USB-2-LCD

### 2.1. Mapeo de Puertos y Sensores Conectados al ZL6

| Puerto | Sensor | Variables Medidas | Unidades | Función Agronómica / Ambiental |
| :--- | :--- | :--- | :--- | :--- |
| **Port 1** | **ATMOS 14** | Temp. Aire, Humedad Relativa, Presión de Vapor | °C, RH %, kPa | Microclima, cálculo de VPD y $ET_o$ |
| **Port 2** | **TEROS 12** | Contenido Volumétrico de Agua (VWC), Temp. Suelo, EC | m³/m³, °C, mS/cm | Humedad, temperatura y salinidad del suelo |
| **Port 3** | **ATMOS 22** | Vel. Viento, Dirección Viento, Ráfaga, Inclinación X/Y | m/s, °, m/s, ° | Dinámica eólica y ventilación |
| **Port 4** | **PYR** | Radiación Solar Global | W/m² | Balance de energía y evapotranspiración |
| **Port 5** | **QSO-S** | Radiación Fotosintéticamente Activa (PPFD) | µmol·m⁻²·s⁻¹ | Radiación disponible para fotosíntesis |
| **Port 6** | **TEROS 21** | Potencial Mátrico, Temp. Suelo | kPa, °C | Disponibilidad hídrica y estrés hídrico |
| **Port 7** | **Batería** | Porcentaje de Batería, Voltaje | %, mV | Diagnóstico de alimentación del logger |
| **Port 8** | **Barómetro**| Presión de Referencia, Temp. Interna Logger | kPa, °C | Diagnóstico de hardware y calibración |

### 2.2. Dispositivos Autónomos Adicionales

| Dispositivo | Variables Medidas | Rangos y Precisión | Características |
| :--- | :--- | :--- | :--- |
| **EL-USB-2-LCD / EL-USB-2-LCD+** | Temp. Aire, Humedad Relativa, Punto de Rocío | Temp: -35 °C a +80 °C (±0.55 °C)<br>RH: 0% a 100% (±2.25%) | Datalogger USB autónomo, pantalla LCD, >16,000 lecturas, IP67 |

### 2.3. Variables Clave para el Balance de Energía (BREB)

| Sensor / Instrumento | Variable Medida | Ubicación / Altura | Símbolo | Unidades | Función en el Balance |
| :--- | :--- | :--- | :---: | :---: | :--- |
| **Piranómetro / Radiómetro** | Radiación Solar / Neta | Dosel superior ($z = 1.8\text{ m}$) | $R_s / R_n$ | $\text{W/m}^2$ | Entrada de energía radiativa total |
| **Sensor T/HR Nivel 1** | Temp. y Humedad Relativa | Nivel inferior ($z_1 = 0.2\text{ m}$) | $T_1, HR_1$ | $^\circ\text{C}, \%$ | Gradiente inferior ($\Delta T, \Delta e$) |
| **Sensor T/HR Nivel 2** | Temp. y Humedad Relativa | Nivel superior ($z_2 = 1.2\text{ m}$) | $T_2, HR_2$ | $^\circ\text{C}, \%$ | Gradiente superior ($\Delta T, \Delta e$) |
| **Placa de Flujo de Calor ($G$)** | Flujo de Calor en Suelo | Subsuperficial ($z_g = -0.2\text{ m}$) | $G$ | $\text{W/m}^2$ | Energía conducida al/del sustrato |
| **Barómetro** | Presión Atmosférica | Interior del invernadero | $P$ | $\text{kPa}$ | Constante psicrométrica local ($\gamma$) |

---

## 3. Estrategia de Limpieza y Preprocesamiento de Datos

* **Variables Conservadas (Dataset de Análisis):**
  * `Timestamp`
  * `Port 1` (ATMOS 14): Temp. Aire (°C), RH (%), Presión de Vapor (kPa)
  * `Port 2` (TEROS 12): VWC (m³/m³), Temp. Suelo (°C), Bulk EC (mS/cm)
  * `Port 3` (ATMOS 22): Vel. Viento (m/s), Dirección (°), Ráfaga (m/s)
  * `Port 4` (PYR): Radiación Solar (W/m²)
  * `Port 5` (QSO-S): PPFD (µmol·m⁻²·s⁻¹)
  * `Port 6` (TEROS 21): Potencial Mátrico (kPa), Temp. Suelo (°C)
  * `EL-USB-2-LCD`: Temp. Aire (°C), RH (%), Punto de Rocío (°C)[cite: 235]

* **Variables Filtradas (Diagnóstico de Hardware):**
  * `ATMOS 22 X/Y-axis Level` (Nivelación horizontal del anemómetro).
  * `Battery Percent` y `Battery Voltage` (Estado de carga del ZL6).
  * `Reference Pressure` y `Logger Temperature` (Diagnóstico interno del ZL6).

---

## 4. Flujo de Procesamiento Computacional (Pipeline)

```text
[1. Datos Crudos (.txt/.xlsx)] ──► Carga de series temporales ZL6 y EL-USB-2-LCD
            │
            ▼
[2. Control y Sincronización] ───► Normalización (pd.to_numeric) y alineación temporal (pd.merge_asof)
            │
            ▼
[3. Resampling Temporal] ────────► Agregación regular a 30 min ('.resample('30min').mean()')
            │
            ▼
[4. Cálculos Psicrométricos] ────► Presión de vapor e_s(T), e_a(T, HR), VPD y gradientes (ΔT, Δe)
            │
            ▼
[5. Radiación Neta y Flujo G] ───► Estimación de R_n (FAO-56) y G en función de R_n (c_día = 0.15, c_noche = 0.50)
            │
            ▼
[6. Razón de Bowen (β)] ─────────► Estimacion singularidad de Bowen (-1.25 < β < -0.75) y umbrales físicos
            │
            ▼
[7. Partición de Flujos] ────────► Despeje de Calor Latente (LE) y Calor Sensible (H)
            │
            ▼
[8. Conversión a ETa] ───────────► Transformación de W/m² a mm/intervalo vía λ y Δt (30 min)
            │
            ▼
[9. Agregación Diaria] ──────────► Integración de la lámina acumulada (.resample('1D').sum())
```

---

## 5. Apéndice: Glosario de Variables y Parámetros

### 5.1. Flujos de Energía y Evapotranspiración

| Símbolo | Variable / Parámetro | Descripción | Unidades |
| :---: | :--- | :--- | :---: |
| $R_n$ | Radiación Neta | Balance neto de energía radiativa de onda corta y larga. | $\text{W/m}^2$ |
| $R_s$ | Radiación Solar Incidente | Radiación solar global de onda corta. | $\text{W/m}^2$ |
| $G$ | Flujo de Calor en el Suelo | Energía conducida hacia o desde el suelo/sustrato. | $\text{W/m}^2$ |
| $H$ | Flujo de Calor Sensible | Transferencia de calor por gradiente térmico aire-superficie. | $\text{W/m}^2$ |
| $LE$ | Flujo de Calor Latente | Energía consumida en el cambio de fase del agua a vapor. | $\text{W/m}^2$ |
| $ET_a$ | Evapotranspiración Real Subdiaria | Lámina de agua evaporada en el intervalo ($\Delta t$). | $\text{mm/intervalo}$ |
| $ET_{a,d}$ | Evapotranspiración Real Diaria | Integración acumulada de $ET_a$ en 24 horas. | $\text{mm/día}$ |

### 5.2. Variables Microclimáticas y Gradientes

| Símbolo | Variable / Parámetro | Descripción | Unidades |
| :---: | :--- | :--- | :---: |
| $T_1, T_2$ | Temperatura del Aire | Temperatura medida a las alturas $z_1$ (inferior) y $z_2$ (superior). | $^\circ\text{C}$ |
| $\Delta T$ | Gradiente de Temperatura | Diferencia vertical de temperatura ($T_2 - T_1$). | $^\circ\text{C}$ |
| $HR_1, HR_2$ | Humedad Relativa | Humedad relativa medida a las alturas $z_1$ y $z_2$. | $\%$ |
| $e_s(T)$ | Presión de Vapor Sat. | Máxima presión de vapor a la temperatura $T$. | $\text{kPa}$ |
| $e_{s1}, e_{s2}$ | Presión Actual de Vapor | Presión parcial de vapor en las alturas $z_1$ y $z_2$. | $\text{kPa}$ |
| $\Delta e$ | Gradiente Presión Vapor | Diferencia vertical de presión de vapor ($e_2 - e_1$). | $\text{kPa}$ |
| $VPD$ | Deficit Presión de Vapor | Capacidad secante del aire ($e_s - e$). | $\text{kPa}$ |
| $P$ | Presión Atmosférica | Presión barométrica local. | $\text{kPa}$ |

### 5.3. Parámetros Físicos y Termodinámicos

| Símbolo | Variable / Parámetro | Descripción | Unidades / Valor |
| :---: | :--- | :--- | :---: |
| $\beta$ | Razón de Bowen | Relación $H / LE = \gamma \cdot (\Delta T / \Delta e)$. | Adimensional |
| $\beta_{\text{qc}}$ | Razón de Bowen Filtrada | Valor de $\beta$ tras control de calidad (QA/QC). | Adimensional |
| $\gamma$ | Constante Psicrométrica | Relación entre temperatura y presión de vapor. | $\text{kPa/}^\circ\text{C}$ ($\approx 0.049$) |
| $\lambda$ | Calor Latente de Vaporización | Energía necesaria para evaporar $1\text{ kg}$ de agua. | $\text{J/kg}$ ($2.45 \times 10^6$) |
| $c_p$ | Calor Específico del Aire | Calor para elevar $1^\circ\text{C}$ la temperatura de $1\text{ kg}$ de aire. | $\text{MJ}/(\text{kg}\cdot^\circ\text{C})$ ($1.013 \times 10^{-3}$) |
| $\epsilon$ | Relación de Masas Molares | Coeficiente $M_w / M_a$. | Adimensional ($0.622$) |
| $c_{\text{day}}$ | Coeficiente Diurno $G$ | Fracción de $R_n$ transferida al suelo durante el día. | Adimensional ($0.15$) |
| $c_{\text{night}}$ | Coeficiente Nocturno $G$ | Fracción de $R_n$ transferida al suelo durante la noche. | Adimensional ($0.50$) |

### 5.4. Variables Espaciotemporales

| Símbolo | Variable / Parámetro | Descripción | Unidades |
| :---: | :--- | :--- | :---: |
| $\text{frec}$ | Frecuencia de Muestreo | Intervalo de agregación temporal (ej. `'30min'`). | Cadena (`str`) - DataFrame|
| $\Delta t$ | Intervalo de Tiempo | Ancho de la ventana temporal en minutos. | $\text{min}$ |
| $\Delta t_{\text{seg}}$ | Intervalo en Segundos | Ancho de la ventana temporal ($\Delta t \times 60$). | $\text{s}$ |
| $z$ | Altitud Topográfica | Elevación geográfica del sitio experimental. | $\text{msnm}$ ($\approx 2600$) |
| $z_1, z_2$ | Alturas de Sensores | Niveles verticales de medición en el invernadero. | $\text{m}$ |


In [36]:
import pandas as pd

In [37]:
def leer_psicrometricos_txt(ruta_archivo, separador = ','):
    """
    Lee un archivo .txt con datos psicrométricos y los carga en una variable (DataFrame)
    para poder analizar la fecha y hora, temperatura, humedad y punto de rocío por separado.
    
    'separador' puede ser ',' (coma), '\t' (tabulador) o ';' (punto y coma)... depende mucho del archivo fuente
    En este caso, el psicrometrico separa los datos por comas
    """
    try:
        # Lee el texto estructurado como una tabla, usando el separador ya definido
        df = pd.read_csv(ruta_archivo, sep=separador)
        df["Time"] = pd.to_datetime(df["Time"]).dt.floor("10min")
        print(f"-> Archivo '{ruta_archivo}' leído exitosamente.")
        return df
    except Exception as e:
        print(f"Error al leer el archivo psicrométrico: {e}")
        return None

In [38]:
def leer_estacion_excel(ruta_archivo, hoja = "Sheet1"):
    """
    Lee un archivo Excel con datos psicrométricos y los carga en un DataFrame.
    'hoja' puede ser el número de la pestaña (0 para la primera hasta n-1) o su nombre exacto (ej. "Datos1").
    """
    try:
        # Lee el xlsx estructurado como una tabla, usando el separador ya definido
        df = pd.read_excel(ruta_archivo, sheet_name = hoja)
        print(f"-> Archivo Excel '{ruta_archivo}' leído exitosamente.")
        return df
    except Exception as e:
        print(f"Error al leer el archivo Excel: {e}")
        return None

In [39]:
# OPCIONAL -> ACOTAMIENTO POR FECHAS Y HORAS
"""
Si los datos de entrada se requiere acotarlos por alguna razon, se puede usar esta funcion
"""

def acotar_df_por_fecha_hora(df_input, fecha_inicio, fecha_final, columna_tiempo):
    """
    Filtra un DataFrame entre un rango de fecha y hora específico, como por ejemplo dia de siembra y dia de cosecha.
    
    Parámetros:
    - df: DataFrame de pandas que contiene tus datos.
    - fecha_inicio: Texto con formato 'YYYY-MM-DD HH:MM:SS'
    - fecha_final: Texto con formato 'YYYY-MM-DD HH:MM:SS'
    - columna_tiempo: Nombre de la columna de tiempo.
    
    Retorna:
    - Un nuevo DataFrame con los datos acotados.
    """
    
    # Se hace una copia para no modificar el DataFrame original por accidente... por si acaso, aunque se puede modificar el df de entrada
    df_filtrado = df_input.copy()
    
    # Asegurar que la columna de tiempo esté en formato datetime, formato util para manipulacion de tiempos y evitar que sea simplemente una cadena de caracteres
    df_filtrado[columna_tiempo] = pd.to_datetime(df_filtrado[columna_tiempo])
    
    # Convertir los límites de texto a formato datetime (incluyendo horas)
    inicio = pd.to_datetime(fecha_inicio)
    fin = pd.to_datetime(fecha_final)
    
    # Aplicar el filtro de rango (mayor o igual al inicio y menor o igual al fin)
    condicion = (df_filtrado[columna_tiempo] >= inicio) & (df_filtrado[columna_tiempo] <= fin)
    df_acotado = df_filtrado[condicion]
    
    print(f"-> Rango aplicado: de {inicio} a {fin}")
    print(f"-> Filas originales: {len(df_input)} | Filas después del acote: {len(df_acotado)}")
    
    return df_acotado

### &#x1F527; ESpecificar la ruta del archivo de texto que contiene los datos de los psicrométricos y anotar fecha inicial y final

In [40]:
ruta_psi1 = '/home/alejandro/labtesis/DATA - acotada/datos_psicrometrico1_completos.txt'
ruta_psi2 = '/home/alejandro/labtesis/DATA - acotada/datos_psicrometrico2_completos.txt'

ini = '2026-04-20 00:00:00' #Siembra
fin = '2026-07-30 00:00:00' #Cosecha

df_ps1 = acotar_df_por_fecha_hora(leer_psicrometricos_txt(ruta_psi1),ini,fin,'Time')
df_ps2 = acotar_df_por_fecha_hora(leer_psicrometricos_txt(ruta_psi2),ini,fin,'Time')
# Para listar las columnas se puede usar df_ps1.colums
# Para imprimir cualquiera de esas columnas df_ps1['nomcolumna']

-> Archivo '/home/alejandro/labtesis/DATA - acotada/datos_psicrometrico1_completos.txt' leído exitosamente.
-> Rango aplicado: de 2026-04-20 00:00:00 a 2026-07-30 00:00:00
-> Filas originales: 14487 | Filas después del acote: 14487
-> Archivo '/home/alejandro/labtesis/DATA - acotada/datos_psicrometrico2_completos.txt' leído exitosamente.
-> Rango aplicado: de 2026-04-20 00:00:00 a 2026-07-30 00:00:00
-> Filas originales: 19415 | Filas después del acote: 14546


In [41]:
df_ps1

,EL 1,Time,Celsius(°C),Humidity(%rh),Dew Point(°C),Serial Number
0,1,2026-04-20 00:00:00,12.0,90.5,10.5,64017443.0
1,2,2026-04-20 00:10:00,12.5,91.0,11.1,NaN
2,3,2026-04-20 00:20:00,12.5,91.5,11.2,NaN
3,4,2026-04-20 00:30:00,13.0,90.0,11.4,NaN
4,5,2026-04-20 00:40:00,13.0,90.5,11.5,NaN
...,...,...,...,...,...,...
14482,14483,2026-07-29 13:30:00,31.0,37.5,14.8,NaN
14483,14484,2026-07-29 13:40:00,31.0,43.0,17.0,NaN
14484,14485,2026-07-29 13:50:00,33.0,55.5,22.9,NaN
14485,14486,2026-07-29 14:00:00,29.5,79.0,25.5,NaN


In [42]:
df_ps2

,EL 6,Time,Celsius(°C),Humidity(%rh),Dew Point(°C),Serial Number
0,1,2026-04-20 00:00:00,12.0,91.5,10.7,NaN
1,2,2026-04-20 00:10:00,12.5,92.0,11.2,NaN
2,3,2026-04-20 00:20:00,12.5,93.0,11.4,NaN
3,4,2026-04-20 00:30:00,12.5,91.5,11.2,NaN
4,5,2026-04-20 00:40:00,12.5,92.0,11.2,NaN
...,...,...,...,...,...,...
14541,14542,2026-07-29 23:20:00,15.0,73.0,10.2,NaN
14542,14543,2026-07-29 23:30:00,15.0,71.5,9.9,NaN
14543,14544,2026-07-29 23:40:00,15.0,73.0,10.2,NaN
14544,14545,2026-07-29 23:50:00,14.5,75.0,10.1,NaN


### &#x1F527; Especificar la ruta del archivo excel que contiene los datos de la estación meteorológica

In [43]:
ruta_est = '/home/alejandro/labtesis/DATA - acotada/z6-13472_acotado_01Abr_14Jul.xlsx'
df_est = leer_estacion_excel(ruta_est)
df_est
# Para ver unicamente los valores numericos puedo usar la funcion .iloc[fila:] ... por ejemplo df_est.iloc[2:]

-> Archivo Excel '/home/alejandro/labtesis/DATA - acotada/z6-13472_acotado_01Abr_14Jul.xlsx' leído exitosamente.


,z6-13472,Port 1,Port 1.1,Port 1.2,Port 2,Port 2.1,Port 2.2,Port 3,Port 3.1,Port 3.2,...,Port 3.4,Port 3.5,Port 4,Port 5,Port 6,Port 6.1,Port 7,Port 7.1,Port 8,Port 8.1
0,Records: 30238,ATMOS 14,ATMOS 14,ATMOS 14,TEROS 12,TEROS 12,TEROS 12,ATMOS 22,ATMOS 22,ATMOS 22,...,ATMOS 22,ATMOS 22,PYR,QSO-S,TEROS 21,TEROS 21,Battery,Battery,Barometer,Barometer
1,Timestamp,°C Air Temperature,RH Relative Humidity,kPa Atmospheric Pressure,m³/m³ Water Content,°C Soil Temperature,mS/cm Bulk EC,° Wind Direction,m/s Wind Speed,m/s Gust Speed,...,° X-axis Level,° Y-axis Level,W/m² Solar Radiation,µmol·m⁻²·s⁻¹ PPFD,kPa Matric Potential,°C Soil Temperature,% Battery Percent,mV Battery Voltage,kPa Reference Pressure,°C Logger Temperature
2,2026-04-01 00:00:00,14.1,0.908725,75.36,0.13214,16.4,0.052,186,0.04,0.07,...,-1.6,0.3,0,0,-10.700289,17.6,100,7908,75.3,13.98
3,2026-04-01 00:05:00,14.16,0.911394,75.36,0.132411,16.4,0.051761,239,0.04,0.06,...,-1.6,0.3,0,0,-10.700289,17.6,100,7893,75.3,14.06
4,2026-04-01 00:10:00,14.18,0.908976,75.35,0.132373,16.36,0.052,171,0.03,0.06,...,-1.7,0.3,0,0,-10.700289,17.58,100,7901,75.28,14.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30235,2026-07-14 23:35:00,12.08,0.881507,75.53,0.341839,18.9,0.028774,213,0.08,0.14,...,2.5,1.8,0,0,-0.097993,18.6,100,7971,75.46,12.19
30236,2026-07-14 23:40:00,12.12,0.888391,75.52,0.3418,18.9,0.028774,243,0.07,0.13,...,2.3,1.7,0,0,-0.097993,18.5,100,7974,75.46,12.25
30237,2026-07-14 23:45:00,12.18,0.89194,75.52,0.341916,18.9,0.028774,219,0.08,0.2,...,2.5,1.7,0,0,-0.097993,18.5,100,7972,75.45,12.27
30238,2026-07-14 23:50:00,12.2,0.891471,75.52,0.341877,18.9,0.028973,223,0.07,0.11,...,2.4,1.8,0,0,-0.097993,18.5,100,7975,75.45,12.22


In [44]:
df_est = acotar_df_por_fecha_hora(df_est.iloc[2:],ini,fin,'z6-13472')
df_est

-> Rango aplicado: de 2026-04-20 00:00:00 a 2026-07-30 00:00:00
-> Filas originales: 30238 | Filas después del acote: 24766


,z6-13472,Port 1,Port 1.1,Port 1.2,Port 2,Port 2.1,Port 2.2,Port 3,Port 3.1,Port 3.2,...,Port 3.4,Port 3.5,Port 4,Port 5,Port 6,Port 6.1,Port 7,Port 7.1,Port 8,Port 8.1
5474,2026-04-20 00:00:00,12.16,0.919943,75.42,0.312475,17.94,0.039994,221,0.07,0.1,...,2.5,1.5,0,0,-0.097993,16.2,100,7974,75.37,12.29
5475,2026-04-20 00:05:00,12.24,0.922137,75.41,0.312513,17.9,0.039994,258,0.05,0.09,...,2.4,1.5,0,0,-0.097993,16.2,100,7961,75.36,12.37
5476,2026-04-20 00:10:00,12.26,0.929341,75.41,0.312552,17.9,0.039811,268,0.06,0.09,...,2.4,1.3,0,0,-0.097993,16.2,100,7968,75.36,12.45
5477,2026-04-20 00:15:00,12.32,0.931969,75.41,0.312513,17.9,0.039994,239,0.07,0.12,...,2.4,1.5,0,0,-0.097993,16.2,100,7969,75.36,12.43
5478,2026-04-20 00:20:00,12.44,0.93574,75.41,0.312436,17.8,0.039994,228,0.06,0.11,...,2.4,1.4,0,0,-0.097993,16.2,100,7968,75.36,12.48
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30235,2026-07-14 23:35:00,12.08,0.881507,75.53,0.341839,18.9,0.028774,213,0.08,0.14,...,2.5,1.8,0,0,-0.097993,18.6,100,7971,75.46,12.19
30236,2026-07-14 23:40:00,12.12,0.888391,75.52,0.3418,18.9,0.028774,243,0.07,0.13,...,2.3,1.7,0,0,-0.097993,18.5,100,7974,75.46,12.25
30237,2026-07-14 23:45:00,12.18,0.89194,75.52,0.341916,18.9,0.028774,219,0.08,0.2,...,2.5,1.7,0,0,-0.097993,18.5,100,7972,75.45,12.27
30238,2026-07-14 23:50:00,12.2,0.891471,75.52,0.341877,18.9,0.028973,223,0.07,0.11,...,2.4,1.8,0,0,-0.097993,18.5,100,7975,75.45,12.22


## &#x1F9E0; Fundamento Teórico: Balance de Energía

Una vez cargados y estructurados los conjuntos de datos de los archivos de entrada, se procede a revisar el fundamento teórico del balance radiativo en el invernadero. Este proceso se fundamenta en la **Primera Ley de la Termodinámica** (Principio de Conservación de la Energía), expresado de forma general como:

$$\text{Entradas} = \text{Salidas} + \Delta S$$

---

### Componentes Generales del Balance Energético

Para un sistema cerrado o semiabierto como un invernadero, los componentes de esta ecuación se desglosan de la siguiente manera:

- **Entradas ($\text{Entradas}$):**
  - **Radiación solar incidente ($R_s$):** Fuente primaria de energía de onda corta que atraviesa la cubierta del invernadero.
  - **Calefacción suplementaria:** Energía añadida en caso de existir sistemas activos de control térmico.
- **Salidas ($\text{Salidas}$):**
  - **Radiación reflejada y emitida:** Fracción de la radiación reflejada por la cubierta o la canopia (albedo) y emisión de radiación térmica de onda larga ($R_L$) hacia el exterior.
  - **Flujo de calor sensible ($H$):** Transferencia de energía térmica por convección entre el aire interno, la superficie vegetal, la estructura y el exterior (acelerado por ventilación).
  - **Flujo de calor latente ($LE$):** Energía exportada por el aire a través de los procesos de evaporación del suelo y transpiración de las plantas (evapotranspiración).
  - **Flujo de calor en el suelo ($G$):** Conducción de energía térmica transferida hacia las capas profundas del suelo (o liberada desde el suelo hacia el interior en momentos de enfriamiento).
- **Almacenamiento ($\Delta S$):**
  - **Masa de aire y estructura:** Energía retenida como cambio de temperatura y humedad en el volumen de aire interno, la estructura física (arcos, soportes) y la capa superficial del suelo.
  - **Biomasa vegetal:** Calor almacenado internamente por la vegetación.
  - **Fotosíntesis:** Fracción de energía fijada en enlaces químicos (generalmente representa $< 2\%$ de la radiación incidente, por lo que se desprecia en el balance térmico).

---

### Deducción Paso a Paso del Balance Radiativo

#### Paso 1: Ecuación fundamental de conservación
$$\text{Entradas} - \text{Salidas} = \Delta S \implies \text{Entradas} = \text{Salidas} + \Delta S$$

#### Paso 2: Identificación de los flujos físicos
- **Entradas radiativas:** $R_{s\downarrow}$ (onda corta entrante) + $R_{L\downarrow}$ (onda larga entrante).
- **Salidas energéticas:** $R_{s\uparrow}$ (onda corta reflejada) + $R_{L\uparrow}$ (onda larga emitida) + $H + LE + G$.

$$\underbrace{(R_{s\downarrow} + R_{L\downarrow})}_{\text{Entradas}} = \underbrace{(R_{s\uparrow} + R_{L\uparrow} + H + LE + G)}_{\text{Salidas}} + \Delta S$$

#### Paso 3: Definición de la Radiación Neta ($R_n$)
Agrupando todos los términos exclusivamente radiativos ($R_n = [R_{s\downarrow} - R_{s\uparrow}] + [R_{L\downarrow} - R_{L\uparrow}]$), la ecuación toma la forma:

$$R_n = H + LE + G + \Delta S$$

#### Paso 4: Simplificación por estado estacionario ($\Delta S \approx 0$)
En escalas de tiempo promediadas (como intervalos horarios o diarios), el cambio de energía almacenada en el aire y la biomasa es insignificante ($\Delta S \approx 0$), reduciendo la expresión a:

$$R_n = H + LE + G$$

#### Paso 5: Convención de signos y balance de energía disponible
Dependiendo del sentido físico asignado a $G$:
- **Suelo como sumidero/salida ($G > 0$ hacia las capas profundas):**
  $$R_n - G = H + LE$$
- **Suelo como fuente de energía (convención de energía disponible agregada):**
  $$R_n + G = H + LE$$

*(Nota: Durante el día, el suelo absorbe calor actuando como pérdida; durante la noche, libera calor hacia la superficie invirtiendo su signo).*

<div style="background-color: #ebf8ff; border-left: 4px solid #3182ce; padding: 12px 16px; border-radius: 6px; margin: 15px 0; color: #2d3748;">
    <p style="color: #2b6cb0; font-weight: bold; margin-top: 0; margin-bottom: 6px;">
        💡 Consideración Teórica para Invernaderos
    </p>
    <p style="margin: 0;">
        En ambientes protegidos, la magnitud de $H$ y $LE$ se ve fuertemente alterada por la tasa de ventilación (natural o mecánica) y el efecto invernadero (atrapamiento de radiación de onda larga por el material de cubierta). Por ello, el cálculo preciso de $\beta$ mediante gradientes verticales de temperatura y humedad relativa es fundamental para estimar la demanda hídrica del cultivo.
    </p>
</div>


---

### Resumen de Flujos de Densidad de Calor

Cada uno de los componentes de flujo de densidad de calor se mide en $\text{W m}^{-2}$ (o $\text{MJ m}^{-2} \text{d}^{-1}$):

| Término | Denominación | Clasificación | Descripción Física |
| :---: | :--- | :---: | :--- |
| **$R_n$** | Radiación Neta | **Entrada / Salida** | Balance neto entre toda la radiación entrante y saliente (onda corta + onda larga). |
| **$G$** | Flujo de Calor en el Suelo | **Salida / Almacenamiento** | Energía conducida desde o hacia la masa del suelo. |
| **$H$** | Flujo de Calor Sensible | **Salida** | Transferencia de energía térmica entre la superficie/vegetación y el aire ambiente por convección. |
| **$LE$** | Flujo de Calor Latente | **Salida** | Energía consumida en los procesos de evapotranspiración del cultivo. |


---

### Acoplamiento con la Razón de Bowen ($\beta$)

Para resolver la partición de la energía disponible ($R_n - G$) en sus componentes sensible ($H$) y latente ($LE$), se utiliza la **Razón de Bowen** ($\beta$), definida como:

$$\beta = \frac{H}{LE} = \gamma \frac{\Delta T}{\Delta e} \rightarrow  \text{Psicrométricos} $$

Sustituyendo $\beta$ en la ecuación del balance radiativo, se obtienen las expresiones algebraicas para calcular $LE$ y $H$ de manera directa a partir de mediciones agrometeorológicas:

$$LE = \frac{R_n - G}{1 + \beta}$$

$$H = \frac{\beta \cdot (R_n - G)}{1 + \beta}$$

Como el objetivo final es comparar la evapotranspiración real del cultivo en el invernadero, el despeje que se toma es el del calor latente, sabiendo que ($LE = \lambda \cdot ETa$).

$$\lambda \cdot ETa = \frac{R_n - G}{1 + \beta}$$

$$ETa = \frac{R_n - G}{(1 + \beta) \cdot \lambda}$$

<div style="background-color: #fff8e6; border-left: 5px solid #ffa000; padding: 15px; border-radius: 4px; margin: 10px 0;">
  <h4 style="margin-top:0; color: #b78103; font-family: sans-serif;">
    ⚠️ Requisito de Medición Gradiente (Método de Razón de Bowen)
  </h4>
  <p style="font-family: sans-serif; color: #333; margin-bottom: 8px;">
    No es posible ni necesario proceder con el cálculo directo de <b>Heat Sensible ($H$)</b> y <b>Heat Latente ($LE$)</b> mediante el método del gradiente si solo se cuenta con un único nivel de medición.
  </p>
  <p style="font-family: sans-serif; color: #555; margin-bottom: 0;">
    Para resolver este método se requiere indispensablemente registrar el gradiente vertical entre dos alturas diferentes ($z_1$ y $z_2$):
  </p>
  <ul style="font-family: sans-serif; color: #444; margin-top: 5px; margin-bottom: 0;">
    <li><b>Diferencia de Temperatura:</b> $\Delta T = T_{z2} - T_{z1}$</li>
    <li><b>Diferencia de Presión de Vapor:</b> $\Delta e = e_{s2} - e_{s1}$ (calculada a partir de $RH$ y $T$)</li>
      Aquí es donde entran en funcionamiento los psicrométricos, ya que al reccoger esas variables, no se hace necesario resolver ecuaciones para el calor latente y el calor sensible
  </ul>
</div>


---



## &#x1F9E0; Fundamento Teórico: Cálculo de la Radiación Neta ($R_n$)

La **Radiación Neta ($R_n$)** es el balance de energía radiante disponible en la superficie del cultivo o suelo dentro del invernadero. Representa la diferencia entre la energía solar entrante absorbida y las pérdidas por emisión térmica de onda larga.

La expresión matemática fundamental para su cálculo es:

$$R_n = R_{ns} - R_{nl}$$

| Término | Denominación |
| :---: | :--- |
| **$R_n$** | Radiación Neta | 
| **$R_{ns}$** | Radiación neta de onda corta | 
| **$R_{nl}$** | Radiación neta de onda larga | 

---

### 1. Desglose de los Componentes Radiativos

#### A. Radiación Neta de Onda Corta ($R_{ns}$)
Es la fracción de energía procedente directamente del Sol (longitudes de onda de $0.3$ a $3\,\mu\text{m}$) que es retenida por la superficie tras descontar la reflexión.

$$R_{ns} = (1 - \alpha) \cdot R_{s\downarrow}$$

* **$R_{s\downarrow}$:** Radiación solar global incidente ($\text{W m}^{-2}$).
* **$\alpha$ (Albedo):** Coeficiente de reflexión de la superficie (adimensional, varía típicamente entre $0.15$ y $0.25$ para vegetación activa).

#### B. Radiación Neta de Onda Larga ($R_{nl}$)
Es la pérdida neta de energía en forma de radiación térmica emitida por la Tierra y la cubierta hacia la atmósfera (longitudes de onda de $3$ a $100\,\mu\text{m}$).

$$R_{nl} = R_{L\uparrow} - R_{L\downarrow}$$

* **$R_{L\uparrow}$:** Radiación térmica de onda larga emitida hacia arriba por el suelo y la vegetación.
* **$R_{L\downarrow}$:** Radiación térmica de onda larga emitida hacia abajo por la atmósfera y el material de la cubierta del invernadero.


<div style="background-color: #f6f8fa; border-left: 4px solid #0969da; padding: 12px 16px; border-radius: 6px; margin: 15px 0; color: #24292f;">
    <p style="color: #0969da; font-weight: bold; margin-top: 0; margin-bottom: 6px;">
        📌 Nota Práctica para el Modelado
    </p>
    <p style="margin: 0;">
        Durante la noche, al no existir radiación solar ($R_{ns} = 0$), la radiación neta se vuelve negativa ($R_n = -R_{nl}$), lo que indica que el suelo y el invernadero están perdiendo calor por radiación térmica hacia el cielo.
    </p>
</div>

---

## Estimación de la Radiación Neta de Onda Larga ($R_{nl}$) según el Método FAO-56

En la práctica agrometeorológica, la medición directa de la radiación de onda larga ($R_{L\uparrow}$ y $R_{L\downarrow}$) mediante piregeómetros es costosa y poco común. Por ello, la metodología estandarizada del **Boletín FAO-56 (Allen et al., 1998)** propone estimar la radiación neta de onda larga saliente ($R_{nl}$) a partir de variables meteorológicas de rutina: **temperatura del aire**, **presión de vapor (humedad)** y **radiación solar relativa (nubosidad)**.

---

### Ecuación General de FAO-56

La expresión matemática estandarizada para el cálculo de $R_{nl}$ es:

$$R_{nl} = \sigma \cdot \left[ \frac{T_{\text{máx, K}}^4 + T_{\text{mín, K}}^4}{2} \right] \cdot \left( 0.34 - 0.14 \sqrt{e_a} \right) \cdot \left( 1.35 \frac{R_s}{R_{so}} - 0.35 \right)$$

---

### Desglose de Parámetros y Variables

| Variable / Constante | Nombre y Significado | Unidades Estándar |
| :---: | :--- | :---: |
| **$\sigma$** | **Constante de Stefan-Boltzmann:** $4.903 \times 10^{-9} \text{ MJ K}^{-4} \text{ m}^{-2} \text{ día}^{-1}$ *(o $5.67 \times 10^{-8} \text{ W m}^{-2} \text{ K}^{-4}$ para cálculo instantáneo/horario)*. | $\text{MJ m}^{-2} \text{K}^{-4} \text{d}^{-1}$ / $\text{W m}^{-2} \text{K}^{-4}$ |
| **$T_{\text{máx, K}}$** | Temperatura máxima absoluta del aire en el periodo: $T_{\text{máx, K}} = T_{\text{máx, }^\circ\text{C}} + 273.16$. | $\text{K}$ |
| **$T_{\text{mín, K}}$** | Temperatura mínima absoluta del aire en el periodo: $T_{\text{mín, K}} = T_{\text{mín, }^\circ\text{C}} + 273.16$. | $\text{K}$ |
| **$e_a$** | **Presión actual de vapor:** Medida o calculada a partir de la humedad relativa ($HR$). | $\text{kPa}$ |
| **$R_s$** | **Radiación solar global medida** que llega a la superficie. | $\text{MJ m}^{-2} \text{d}^{-1}$ / $\text{W m}^{-2}$ |
| **$R_{so}$** | **Radiación solar en cielo despejado:** Valor teórico máximo de radiación solar para la ubicación y día del año. | $\text{MJ m}^{-2} \text{d}^{-1}$ / $\text{W m}^{-2}$ |
| **$\frac{R_s}{R_{so}}$** | **Radiación solar relativa:** Coeficiente de transmisividad atmosférica / cobertura nubosa (acotado típicamente en el intervalo $[0.33, 1.0]$). | Adimensional |

---

### Análisis Físico de los Tres Términos Modificadores

El método FAO-56 descompone la pérdida de onda larga en el producto de tres componentes físicos:

1. **Término Térmico $\left( \sigma \cdot T^4 \right)$:**  
   Calcula la emisión media de cuerpo negro según la **Ley de Stefan-Boltzmann**, promediando las temperaturas extremas absolutas registradas en la superficie.

2. **Término de Humedad Atmosférica $\left( 0.34 - 0.14 \sqrt{e_a} \right)$:**  
   Modula la emisión en función del vapor de agua en el aire. El agua en estado gaseoso absorbe la radiación de onda larga y la reemite hacia la superficie. A mayor presión de vapor ($e_a$), menor es la pérdida neta de energía ($R_{nl}$).

3. **Término de Nubosidad / Cobertura $\left( 1.35 \frac{R_s}{R_{so}} - 0.35 \right)$:**  
   Ajusta el flujo radiativo según la presencia de nubes. Las nubes actúan como una barrera que devuelve radiación térmica a la tierra. Un día despejado ($\frac{R_s}{R_{so}} \approx 1$) maximiza la pérdida neta $R_{nl}$, mientras que un día muy nublado la reduce sensiblemente.

<div style="background-color: #faf5ff; border-left: 4px solid #805ad5; padding: 12px 16px; border-radius: 6px; margin: 15px 0; color: #2d3748;">
    <p style="color: #44337a; font-weight: bold; margin-top: 0; margin-bottom: 6px;">
        💡 Adaptación del Término de Nubosidad en Invernaderos
    </p>
    <p style="margin: 0;">
        En el interior de un invernadero, la relación $\frac{R_s}{R_{so}}$ debe considerar no solo la nubosidad exterior, sino también la <strong>transmisividad de la cubierta plástica/vidrio</strong> ($\tau$). La cubierta retiene gran parte de la radiación $R_{L\uparrow}$ (efecto térmico), por lo que el valor efectivo de $R_{nl}$ en el microclima protegido suele ser menor que al aire libre.
    </p>
</div>


<div style="background-color: #fffaf0; border-left: 4px solid #dd6b20; padding: 12px 16px; border-radius: 6px; margin: 15px 0; color: #2d3748;">
    <p style="color: #7b341e; font-weight: bold; margin-top: 0; margin-bottom: 6px;">
        ⚠️ Consideración Metodológica: Estación Meteorológica Interna
    </p>
    <p style="margin: 0;">
        Dado que los datos agrometeorológicos se registran en el <strong>interior del invernadero</strong>, la radiación solar medida ($R_{s,\text{int}}$) está afectada por el coeficiente de transmisividad de la cubierta ($\tau$). Para evitar que el método FAO-56 interprete la opacidad del plástico como nubosidad atmosférica, la fracción de radiación solar relativa se ajusta mediante:
    </p>
    <p style="margin-top: 8px; margin-bottom: 0; text-align: center;">
        $$\left( \frac{R_s}{R_{so}} \right)_{\text{ajustado}} = \min\left( \frac{R_{s,\text{int}}}{\tau \cdot R_{so}}, 1.0 \right)$$
    </p>
    <p style="margin-top: 8px; margin-bottom: 0;">
        Asimismo, las variables de temperatura ($T$) y humedad relativa ($HR$) registradas internamente capturan de forma directa el microclima del canopia, garantizando una estimación adecuada del flujo de emisión térmica del suelo y cultivo ($R_{L\uparrow}$).
    </p>
</div>

---


## Estimación de la Radiación Solar en Cielo Despejado ($R_{so}$)

La **radiación solar en cielo despejado ($R_{so}$)** representa la máxima radiación solar teórica que llega a la superficie terrestre en un día o periodo determinado sin interferencia de nubes. Es un parámetro indispensable en la metodología FAO-56 para determinar la transmisividad atmosférica.

---

### Ecuación Operativa FAO-56

Cuando se conoce la altitud sobre el nivel del mar del sitio de estudio ($z$), $R_{so}$ se calcula a partir de la **radiación extraterrestre ($R_a$)**:

$$R_{so} = (0.75 + 2 \times 10^{-5} \cdot z) \cdot R_a$$

---

### Desglose de Parámetros de $R_{so}$

| Parámetro | Definición | Unidades Estándar |
| :---: | :--- | :---: |
| **$R_{so}$** | Radiación solar en cielo despejado. | $\text{MJ m}^{-2} \text{d}^{-1}$ / $\text{W m}^{-2}$ |
| **$z$** | Altitud sobre el nivel del mar de la estación meteorológica. | $\text{m}$ |
| **$R_a$** | Radiación extraterrestre (energía solar que llega al tope de la atmósfera). | $\text{MJ m}^{-2} \text{d}^{-1}$ / $\text{W m}^{-2}$ |

---

### Cálculo de la Radiación Extraterrestre ($R_a$)

Para un periodo diario, $R_a$ depende exclusivamente de la posición geográfica (latitud $\varphi$) y del día del año ($J$):

$$R_a = \frac{24(60)}{\pi} G_{sc} \cdot d_r \left[ \omega_s \sin(\varphi)\sin(\delta) + \cos(\varphi)\cos(\delta)\sin(\omega_s) \right]$$

Donde:
* **$G_{sc}$:** Constante solar ($0.0820 \text{ MJ m}^{-2} \text{ min}^{-1}$).
* **$d_r$:** Distancia relativa inversa Tierra-Sol: $d_r = 1 + 0.033 \cos\left(\frac{2\pi}{365} J\right)$.
* **$\delta$:** Declinación solar en radianes: $\delta = 0.409 \sin\left(\frac{2\pi}{365} J - 1.39\right)$.
* **$\omega_s$:** Ángulo horario al atardecer: $\omega_s = \arccos\left(-\tan(\varphi)\tan(\delta)\right)$.
* **$\varphi$:** Latitud en radianes ($\text{rad} = \frac{\pi}{180} \cdot \text{grados}$).


<div style="background-color: #f0fff4; border-left: 4px solid #38a169; padding: 12px 16px; border-radius: 6px; margin: 15px 0; color: #2d3748;">
    <p style="color: #22543d; font-weight: bold; margin-top: 0; margin-bottom: 6px;">
        📌 Aplicación en Escala Horaria o Sub-horaria
    </p>
    <p style="margin: 0;">
        Como los datos registrados son de frecuencias cortas (ej. cada 5, 10 o 60 minutos), $R_a$ y $R_{so}$ no se calculan con el ángulo del atardecer $\omega_s$, sino en función del <strong>ángulo horario del centro del intervalo</strong> ($\omega$). Esto permite ajustar dinámicamente la curva teórica del sol durante el transcurso del día.
    </p>
</div>

---

## Cálculo de la Presión Actual de Vapor ($e_a$)

La **presión actual de vapor ($e_a$)** mide la presión parcial ejercida por el agua en estado gaseoso presente en el aire. Es una variable fundamental para cuantificar el déficit de presión de vapor ($VPD$) y ajustar la emisión de radiación de onda larga ($R_{nl}$).

---

### Ecuación de Presión de Vapor de Saturación $e^0(T)$

Primero, se calcula la presión de vapor que el aire tendría si estuviese 100% saturado a una temperatura $T$ ($^\circ\text{C}$) mediante la ecuación de Tetens (FAO-56):

$$e^0(T) = 0.6108 \cdot \exp\left( \frac{17.27 \cdot T}{T + 237.3} \right)$$

*Donde $e^0(T)$ se obtiene en kilopascales ($\text{kPa}$).*

---

### Formas de Determinación de $e_a$ según los Datos Disponibles

Dependiendo de la resolución temporal de tus registros de humedad relativa ($HR$) y temperatura ($T$), se aplica:

#### Registros Instantáneos o Escala Horaria (Ideal para Invernaderos)
Cuando dispones de lecturas continuas de $T$ ($^\circ\text{C}$) y $HR$ ($\%$) en el datalogger:

$$e_a = e^0(T) \cdot \left( \frac{HR}{100} \right)$$

---

### Resumen de Variables para el Cálculo de $e_a$

| Variable | Descripción | Unidad |
| :---: | :--- | :---: |
| **$e_a$** | Presión actual de vapor del aire. | $\text{kPa}$ |
| **$e^0(T)$** | Presión de saturación de vapor a la temperatura $T$. | $\text{kPa}$ |
| **$T_{\text{máx}}, T_{\text{mín}}$** | Temperatura máxima y mínima del aire en el periodo. | $^\circ\text{C}$ |
| **$HR_{\text{máx}}, HR_{\text{mín}}$** | Humedad relativa máxima y mínima en el periodo. | $\%$ |
| **$HR$** | Humedad relativa instantánea o promediada. | $\%$ |

In [45]:
import numpy as np

In [46]:
def calcular_presiones_vapor(temp, hr):
    """
    Calcula la presión de vapor de saturación (e0), la presión actual de vapor (ea)
    y el déficit de presión de vapor (VPD) según la metodología FAO-56.

    Parámetros:
    -----------
    temp : float, pd.Series o np.ndarray
        Temperatura del aire en grados Celsius (°C).
    hr : float, pd.Series o np.ndarray
        Humedad relativa en porcentaje (0 - 100 %).

    Retorna:
    --------
    pd.DataFrame :
        - e0 : Presión de vapor de saturación (kPa).
        - ea : Presión actual de vapor (kPa).
        - vpd: Déficit de presión de vapor (kPa).
    """
    # 1. Presión de vapor de saturación e0 (Ecuación de Tetens / FAO-56)
    e0 = 0.6108 * np.exp((17.27 * temp) / (temp + 237.3))
    
    # 2. Presión actual de vapor ea
    ea = e0 * (hr / 1.0)
    
    # 3. Déficit de presión de vapor (VPD)
    vpd = e0 - ea
    
    # Si las entradas son Series de Pandas, retorna un DataFrame estructurado
    if isinstance(temp, pd.Series):
        return pd.DataFrame({
            'e0_kPa': e0,
            'ea_kPa': ea,
            'vpd_kPa': vpd
        }, index=temp.index)
    
    return e0, ea, vpd

In [47]:
def calcular_Ra_subdiaria(index_datetime, lat_deg, lon_deg, lon_z_deg=None):
    """
    Calcula la radiación extraterrestre (Ra) a escala sub-diaria usando 
    la notación formal FAO-56 (Ecuación 28).

    Parámetros:
    -----------
    index_datetime : pd.DatetimeIndex
        Índice de fechas/horas de los datos.
    lat_deg : float
        Latitud en grados decimales (phi en grados).
    lon_deg : float
        Longitud del sitio en grados decimales (Lm).
    lon_z_deg : float, opcional
        Longitud del meridiano de la zona horaria local (Lz).

    Variables astronómicas (FAO-56):
    --------------------------------
    G_sc  : Constante solar (0.0820 MJ m^-2 min^-1)
    phi   : Latitud en radianes
    J     : Día del año (1 - 365) o día Juliano
    dr    : Distancia relativa inversa Tierra-Sol
    delta : Declinación solar (rad)
    Et    : Ecuación del Tiempo (horas)
    w     : Ángulo horario al centro del intervalo (rad)
    w1    : Ángulo horario al inicio del intervalo (rad)
    w2    : Ángulo horario al final del intervalo (rad)
    """
    if not isinstance(index_datetime, pd.DatetimeIndex):
        index_datetime = pd.DatetimeIndex(index_datetime)

    # 1. Constantes y Latitud (phi)
    G_sc = 0.0820  # Constante solar (MJ m^-2 min^-1)
    phi = np.radians(lat_deg)
    
    # 2. Día del año (J)
    J = index_datetime.dayofyear
    
    # 3. Geometría orbital: dr y delta
    dr = 1 + 0.033 * np.cos((2 * np.pi / 365) * J)
    delta = 0.409 * np.sin((2 * np.pi / 365) * J - 1.39)
    
    # 4. Ecuación del Tiempo (Et)
    b = (2 * np.pi * (J - 81)) / 364
    Sc = 0.1645 * np.sin(2 * b) - 0.1255 * np.cos(b) - 0.025 * np.sin(b)
    
    # 5. Meridianos de referencia: Lz (Zona horaria) y Lm (Medición)
    Lz = round(lon_deg / 15.0) * 15.0 if lon_z_deg is None else lon_z_deg
    Lm = lon_deg
    
    # 6. Hora decimal (t)
    t = index_datetime.hour + index_datetime.minute / 60.0 + index_datetime.second / 3600.0
    
    # 7. Ángulo horario central (w)
    w = (np.pi / 12) * ((t + Sc + (Lz - Lm) / 15.0) - 12)
    
    # 8. Ancho del intervalo (dt en minutos y horas)
    dt_min = (index_datetime[1] - index_datetime[0]).total_seconds() / 60.0 if len(index_datetime) > 1 else 60.0
    dt_h = dt_min / 60.0
    
    # 9. Encuadres del intervalo: w1 y w2
    w_delta = (np.pi / 12) * (dt_h / 2.0)
    w1 = w - w_delta
    w2 = w + w_delta
    
    # 10. Ecuación 28 FAO-56: Integración de Ra (MJ m^-2 por intervalo)
    Ra_MJ = ((12 * 60) / np.pi) * G_sc * dr * (
        (w2 - w1) * np.sin(phi) * np.sin(delta) +
        np.cos(phi) * np.cos(delta) * (np.sin(w2) - np.sin(w1))
    )
    
    # Acotamiento nocturno
    Ra_MJ = np.maximum(0.0, Ra_MJ)
    
    # 11. Conversión a W m^-2 (Potencia por segundo)
    Ra_W = (Ra_MJ * 1e6) / (dt_min * 60.0)
    
    return pd.DataFrame({
        'Ra_MJ_m2': Ra_MJ,
        'Ra_W_m2': Ra_W
    }, index=index_datetime)

In [48]:
def calcular_Rso(Ra, z):
    """
    Calcula la radiación solar en cielo despejado (Rso) según la Ecuación 37 del FAO-56.
    
    Parámetros:
    -----------
    Ra : float, pd.Series o np.ndarray
        Radiación extraterrestre (Ra).
    z  : float
        Altitud sobre el nivel del mar en metros.
    """
    # Factor de transmisión atmosférica en cielo despejado (k_rs)
    k_rs = 0.75 + (2e-5 * z)
    
    return k_rs * Ra

In [49]:
def calcular_Rnl(T_air_C, e_a_kPa, R_s, R_so, dt_min=5.0):
    """
    Calcula la radiación neta de onda larga (Rnl) según el Boletín FAO-56 (Ecuación 39).
    """
    # 1. Convertir a arreglos NumPy para evitar conflictos de alineación
    T_arr = np.asarray(T_air_C, dtype=float)
    ea_arr = np.asarray(e_a_kPa, dtype=float)
    Rs_arr = np.asarray(R_s, dtype=float)
    Rso_arr = np.asarray(R_so, dtype=float)

    # 2. Constante de Stefan-Boltzmann (W m^-2 K^-4)
    sigma = 5.670374e-8
    
    # 3. Temperatura absoluta en Kelvin
    T_K = T_arr + 273.16
    
    # 4. Factor de emisividad neta del aire
    f_emisividad = 0.34 - 0.14 * np.sqrt(np.maximum(0.0, ea_arr))
    
    # 5. Relación Rs / Rso (evitando división por cero en la noche)
    with np.errstate(divide='ignore', invalid='ignore'):
        r_rel = np.where(Rso_arr > 1.0, Rs_arr / Rso_arr, np.nan)
    
    # Acotamiento FAO-56: 0.30 <= Rs/Rso <= 1.00
    r_rel = np.clip(r_rel, 0.30, 1.00)
    
    # Relleno de valores nocturnos (ffill/bfill) usando Series
    idx = T_air_C.index if isinstance(T_air_C, pd.Series) else None
    s_rrel = pd.Series(r_rel, index=idx).ffill().bfill().fillna(0.70)
    
    # 6. Factor de nubosidad (1.35 * Rs/Rso - 0.35)
    f_cloud = 1.35 * s_rrel.values - 0.35
    f_cloud = np.clip(f_cloud, 0.05, 1.00)
    
    # 7. Cálculo final de Rnl
    Rnl_W = sigma * (T_K**4) * f_emisividad * f_cloud
    Rnl_MJ = (Rnl_W * (dt_min * 60.0)) / 1e6
    
    return pd.DataFrame({
        'Rnl_W_m2': Rnl_W,
        'Rnl_MJ_m2': Rnl_MJ
    }, index=idx)

In [50]:
def calcular_Rns(R_s, albedo=0.23, dt_min=None):
    """
    Calcula la Radiación Neta de Onda Corta (Rns) según el Boletín FAO-56 (Ecuación 38).

    Ecuación:
    ---------
    Rns = (1 - alpha) * Rs

    Parámetros:
    -----------
    R_s : float, np.ndarray o pd.Series
        Radiación solar medida o incidente (W/m² o MJ/m²).
    albedo : float, opcional
        Coeficiente de reflexión (adimensional). Por defecto 0.23 (FAO-56).
    dt_min : float, opcional
        Ancho del intervalo temporal en minutos (ej. 5.0). Si se especifica,
        retorna un DataFrame con columnas para W/m² y MJ/m².

    Retorna:
    --------
    pd.DataFrame o pd.Series:
        DataFrame con ambas unidades si se pasa dt_min y R_s es pandas Series,
        o una Series/array en las mismas unidades de entrada si dt_min es None.
    """
    Rns = (1.0 - albedo) * R_s

    if dt_min is not None and isinstance(R_s, pd.Series):
        Rns_W = Rns
        Rns_MJ = (Rns_W * (dt_min * 60.0)) / 1e6
        
        return pd.DataFrame({
            'Rns_W_m2': Rns_W,
            'Rns_MJ_m2': Rns_MJ
        }, index=R_s.index)

    return Rns

In [51]:
def calcular_Rn (Rns,Rnl):
    Rn = Rns - Rnl
    
    # 2. Si se pasa un DataFrame/Series y dt_min, retornamos un DataFrame estructurado
    if dt_min is not None and isinstance(Rns, pd.Series):
        # Asumiendo que las entradas están en W/m²
        Rn_W = Rn
        Rn_MJ = (Rn_W * (dt_min * 60.0)) / 1e6
        
        return pd.DataFrame({
            'Rn_W_m2': Rn_W,
            'Rn_MJ_m2': Rn_MJ
        }, index=Rns.index)
    
    return Rn

In [52]:
df_est[['e0_kPa', 'ea_kPa', 'vpd_kPa']] = calcular_presiones_vapor(
    temp = pd.to_numeric(df_est['Port 1']), 
    hr = pd.to_numeric(df_est['Port 1.1'])
)
#pd.to_numeric lo que hace es asegurar que los datos guardados, sean leidos como numeros y no como una cadena de caracteres... es mejor asegurar

In [53]:
df_est[['Port 1', 'Port 1.1', 'e0_kPa', 'ea_kPa', 'vpd_kPa']]

,Port 1,Port 1.1,e0_kPa,ea_kPa,vpd_kPa
5474,12.16,0.919943,1.417430,1.303955,0.113475
5475,12.24,0.922137,1.424915,1.313966,0.110949
5476,12.26,0.929341,1.426792,1.325976,0.100816
5477,12.32,0.931969,1.432435,1.334985,0.097449
5478,12.44,0.93574,1.443779,1.351002,0.092777
...,...,...,...,...,...
30235,12.08,0.881507,1.409980,1.242907,0.167073
30236,12.12,0.888391,1.413701,1.255918,0.157782
30237,12.18,0.89194,1.419298,1.265928,0.153370
30238,12.2,0.891471,1.421168,1.266930,0.154238


### &#x1F527; Especificar longitud y latitud

In [54]:
lon = -74.089115
lat = 4.636024

df_est['z6-13472'] = pd.to_datetime(df_est['z6-13472'])
df_est.set_index('z6-13472', inplace=True)

df_est[['Ra_MJ_m2', 'Ra_W_m2']] = calcular_Ra_subdiaria(
    df_est.index, 
    lat, 
    lon
)

### &#x1F527; Especificar altura sobre el nivel del mar

In [55]:
z = 2640.0

# 1. Asignar Rso en MJ/m²
df_est['Rso_MJ_m2'] = calcular_Rso(
    pd.to_numeric(df_est['Ra_MJ_m2']),
    z
)

# 2. Asignar Rso en W/m²
df_est['Rso_W_m2'] = calcular_Rso(
    df_est['Ra_W_m2'],
    z
)

### &#x1F527; Colocar el coeficiente de transmisividad de la cubierta plástica del invernadero y verificar la frecuencia de la toma de datos de la estación meteorológica

In [56]:
dt_min = 5.0
coefTrasmPlast = 0.8 # Coeficiente de trasnsmisividad del plastico de cubierta del invernadero

df_est[['Rnl_W_m2','Rnl_MJ_m2']] = calcular_Rnl(
    df_est['Port 1'],
    df_est['ea_kPa'],
    df_est['Port 4']/coefTrasmPlast,
    df_est['Rso_W_m2'],
    dt_min
)

### &#x1F527; En caso de tener el valor del albedo

In [57]:
albedo = 0.23

df_est[['Rns_W_m2','Rns_MJ_m2']] = calcular_Rns(
    df_est['Port 4'],
    albedo,
    dt_min
)

In [58]:
df_est[['Rn_W_m2','Rn_MJ_m2']] = calcular_Rn(
    df_est['Rns_W_m2'],
    df_est['Rnl_W_m2']
)
df_est

,Port 1,Port 1.1,Port 1.2,Port 2,Port 2.1,Port 2.2,Port 3,Port 3.1,Port 3.2,Port 3.3,...,Ra_MJ_m2,Ra_W_m2,Rso_MJ_m2,Rso_W_m2,Rnl_W_m2,Rnl_MJ_m2,Rns_W_m2,Rns_MJ_m2,Rn_W_m2,Rn_MJ_m2
z6-13472,,,,,,,,,,,,,,,,,,,,,
2026-04-20 00:00:00,12.16,0.919943,75.42,0.312475,17.94,0.039994,221,0.07,0.1,11.7,...,0.0,0.0,0.0,0.0,44.599162,0.013380,0.0,0.0,-44.599162,-0.01338
2026-04-20 00:05:00,12.24,0.922137,75.41,0.312513,17.9,0.039994,258,0.05,0.09,11.78,...,0.0,0.0,0.0,0.0,44.497384,0.013349,0.0,0.0,-44.497384,-0.013349
2026-04-20 00:10:00,12.26,0.929341,75.41,0.312552,17.9,0.039811,268,0.06,0.09,11.8,...,0.0,0.0,0.0,0.0,44.328435,0.013299,0.0,0.0,-44.328435,-0.013299
2026-04-20 00:15:00,12.32,0.931969,75.41,0.312513,17.9,0.039994,239,0.07,0.12,11.88,...,0.0,0.0,0.0,0.0,44.230048,0.013269,0.0,0.0,-44.230048,-0.013269
2026-04-20 00:20:00,12.44,0.93574,75.41,0.312436,17.8,0.039994,228,0.06,0.11,11.94,...,0.0,0.0,0.0,0.0,44.063980,0.013219,0.0,0.0,-44.06398,-0.013219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-14 23:35:00,12.08,0.881507,75.53,0.341839,18.9,0.028774,213,0.08,0.14,11.86,...,0.0,0.0,0.0,0.0,3.797038,0.001139,0.0,0.0,-3.797038,-0.001139
2026-07-14 23:40:00,12.12,0.888391,75.52,0.3418,18.9,0.028774,243,0.07,0.13,12.14,...,0.0,0.0,0.0,0.0,3.782337,0.001135,0.0,0.0,-3.782337,-0.001135
2026-07-14 23:45:00,12.18,0.89194,75.52,0.341916,18.9,0.028774,219,0.08,0.2,11.98,...,0.0,0.0,0.0,0.0,3.772619,0.001132,0.0,0.0,-3.772619,-0.001132


In [59]:
df_est.loc['2026-07-14 11:00:00':'2026-07-14 13:00:00', ['Port 4','Ra_W_m2', 'Ra_MJ_m2','Rso_W_m2','Rso_MJ_m2','Rnl_W_m2','Rnl_MJ_m2','Rns_W_m2','Rns_MJ_m2','Rn_W_m2','Rn_MJ_m2']]

,Port 4,Ra_W_m2,Ra_MJ_m2,Rso_W_m2,Rso_MJ_m2,Rnl_W_m2,Rnl_MJ_m2,Rns_W_m2,Rns_MJ_m2,Rn_W_m2,Rn_MJ_m2
z6-13472,,,,,,,,,,,
2026-07-14 11:00:00,188.964844,1209.525901,0.362858,971.007394,0.291302,4.031324,0.001209,145.50293,0.043651,141.471606,0.042441
2026-07-14 11:05:00,163.146973,1217.192459,0.365158,977.162106,0.293149,4.063676,0.001219,125.623169,0.037687,121.559493,0.036468
2026-07-14 11:10:00,163.330078,1224.298446,0.367290,982.866793,0.294860,4.174386,0.001252,125.76416,0.037729,121.589774,0.036477
2026-07-14 11:15:00,179.992676,1230.840480,0.369252,988.118737,0.296436,4.166202,0.001250,138.59436,0.041578,134.428158,0.040328
2026-07-14 11:20:00,294.250488,1236.815448,0.371045,992.915441,0.297875,10.879220,0.003264,226.572876,0.067972,215.693655,0.064708
2026-07-14 11:25:00,246.276855,1242.220505,0.372666,997.254621,0.299176,4.874041,0.001462,189.633179,0.05689,184.759138,0.055428
2026-07-14 11:30:00,301.574707,1247.053079,0.374116,1001.134212,0.300340,11.372408,0.003412,232.212524,0.069664,220.840117,0.066252
2026-07-14 11:35:00,274.841309,1251.310871,0.375393,1004.552367,0.301366,8.345548,0.002504,211.627808,0.063488,203.28226,0.060985
2026-07-14 11:40:00,231.994629,1254.991853,0.376498,1007.507459,0.302252,4.081763,0.001225,178.635864,0.053591,174.554101,0.052366


## &#x1F9E0; Fundamento Teórico: Cálculo de Flujos Turbulentos ($H$ y $LE$) mediante la Razón de Bowen (BREB)

A partir de los psicrómetros **EL 6** y **EL 1** instalados a dos alturas distintas dentro del invernadero ($z_1$ nivel inferior y $z_2$ nivel superior), se mide el gradiente vertical microclimático para separar la energía disponible ($R_n - G$) en **Calor Sensible ($H$)** y **Calor Latente ($LE$)**.

---

###  Procesamiento de Presión de Vapor ($e$) por Nivel

Para cada psicrómetro se calcula la presión de vapor actual ($e$, en $\text{kPa}$) utilizando la Temperatura del Aire ($T$, en $^\circ\text{C}$) y la Humedad Relativa ($HR$, en $\%$):

$$e_o(T) = 0.6108 \cdot \exp\left( \frac{17.27 \cdot T}{T + 237.3} \right)$$

$$e = e_o(T) \cdot \left( \frac{HR}{100} \right)$$

* **Nivel Inferior ($z_1$):** Registra $T_1$ y $e_1$.
* **Nivel Superior ($z_2$):** Registra $T_2$ y $e_2$.

---

###  Cálculo de Gradientes y Razón de Bowen ($\beta$)

Se determinan las diferencias verticales entre alturas:

$$\Delta T = T_2 - T_1$$

$$\Delta e = e_2 - e_1$$

La **Razón de Bowen ($\beta$)** se calcula como:

$$\beta = \gamma \cdot \left( \frac{\Delta T}{\Delta e} \right)$$

---

<div style="background-color: #f0f7ff; border: 1px solid #b6d4fe; border-left: 6px solid #0d6efd; padding: 20px; border-radius: 8px; margin: 20px 0;">

<h3 style="margin-top:0; color: #084298; font-family: sans-serif; display: flex; align-items: center; gap: 8px;">
  📍 Sustentación Física de la Constante Psicrométrica ($\gamma$) para Bogotá
</h3>

<p style="font-family: sans-serif; color: #212529; font-size: 14.5px; line-height: 1.6;">
La constante psicrométrica $\gamma$ representa la relación entre la capacidad calorífica del aire seco y el calor de vaporización del agua. Según el estándar FAO-56, su valor simplificado depende directamente de la <b>presión atmosférica ($P$)</b>:
</p>

$$\gamma = \frac{C_p \cdot P}{\epsilon \cdot \lambda} = 0.000665 \cdot P$$

<p style="font-family: sans-serif; color: #212529; font-size: 14.5px; line-height: 1.6; margin-bottom: 15px;">
Donde $C_p = 1.013 \times 10^{-3} \text{ MJ kg}^{-1} \text{°C}^{-1}$ (calor específico del aire), $\epsilon = 0.622$ (cociente de masas moleculares) y $\lambda \approx 2.45 \text{ MJ kg}^{-1}$ (calor latente de vaporización).
</p>

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; font-size: 13.5px; background-color: #ffffff; border-radius: 6px; overflow: hidden; box-shadow: 0 1px 3px rgba(0,0,0,0.1);">
  <thead>
    <tr style="background-color: #0d6efd; color: #ffffff; text-align: left;">
      <th style="padding: 10px 12px;">Ubicación / Condición</th>
      <th style="padding: 10px 12px;">Altitud</th>
      <th style="padding: 10px 12px;">Presión Atmosférica ($P$)</th>
      <th style="padding: 10px 12px;">Constante Psicrométrica ($\gamma$)</th>
      <th style="padding: 10px 12px;">Efecto Físico</th>
    </tr>
  </thead>
  <tbody>
    <tr style="border-bottom: 1px solid #dee2e6;">
      <td style="padding: 10px 12px; font-weight: bold; color: #495057;">Nivel del Mar</td>
      <td style="padding: 10px 12px;">0 m.s.n.m.</td>
      <td style="padding: 10px 12px;">101.3 kPa</td>
      <td style="padding: 10px 12px; font-weight: bold;">0.0673 kPa °C⁻¹</td>
      <td style="padding: 10px 12px; color: #6c757d;">Aire estándar a nivel del mar.</td>
    </tr>
    <tr style="background-color: #e7f1ff;">
      <td style="padding: 10px 12px; font-weight: bold; color: #084298;">Invernadero (Bogotá)</td>
      <td style="padding: 10px 12px; font-weight: bold; color: #084298;">2550 m.s.n.m.</td>
      <td style="padding: 10px 12px; font-weight: bold; color: #084298;">75.4 kPa (Port 1.2)</td>
      <td style="padding: 10px 12px; font-weight: bold; color: #0d6efd;">0.0501 ≈ 0.051 kPa °C⁻¹</td>
      <td style="padding: 10px 12px; font-weight: bold; color: #084298;">Aire ~25.5% menos denso.</td>
    </tr>
  </tbody>
</table>

<div style="margin-top: 15px; padding: 10px 14px; background-color: #ffffff; border-left: 4px solid #0dcaf0; border-radius: 4px; font-family: sans-serif; font-size: 13.5px; color: #055160;">
  <b>💡 Importancia Bioclimática:</b> Dado que la columna de aire en la Sabana de Bogotá ejerce menor presión, el aire ofrece menor resistencia a la masa de vapor y requiere menos energía para elevar su temperatura. Usar $\gamma \approx 0.051 \text{ kPa }^\circ\text{C}^{-1}$ previene sobreestimaciones en el flujo de calor sensible ($H$) dentro del invernadero.
</div>

</div>

---

###  Cierre del Balance Energético ($LE$ y $H$)

Con la energía neta disponible ($R_n - G$) y la razón $\beta$, los componentes del balance se calculan de forma directa:

1. **Flujo de Calor Latente / Evapotranspiración ($LE$):**
   $$LE = \frac{R_n - G}{1 + \beta}$$

2. **Flujo de Calor Sensible ($H$):**
   $$H = \frac{\beta}{1 + \beta} \cdot (R_n - G) = \beta \cdot LE$$

<div style="background-color: #fff3cd; border: 1px solid #ffe69c; border-left: 5px solid #ffc107; padding: 12px 16px; border-radius: 6px; margin-top: 15px; font-family: sans-serif; font-size: 13.5px; color: #664d03;">
  <b>⚠️ Filtro de Calidad Operacional (Singularidad Matemática):</b> Cuando la temperatura o la humedad relativa cambian de tendencia durante las transiciones de amanecer o atardecer, $\beta$ puede aproximarse a $-1$, haciendo que la división $(1 + \beta)$ tienda a cero. Se recomienda omitir o aplicar interpolación lineal en los registros donde $-1.25 < \beta < -0.75$.
</div>

---

##  Cancelación de la Altura ($\Delta z$)

Como ya se habia dicho, la *Razón de Bowen* ($\beta$) relaciona el flujo de calor sensible ($H$) y el flujo de calor latente ($LE$) mediante la relación de sus gradientes verticales:

$$H = -\rho c_p K_h \frac{\Delta T}{\Delta z}$$

$$LE = -\frac{\rho L_v \epsilon}{P} K_w \frac{\Delta e_a}{\Delta z}$$

Al dividir ambos flujos ($\beta = \frac{H}{LE}$) y asumir la equivalencia en los coeficientes de transporte turbulento ($K_h = K_w$), la distancia vertical ($\Delta z$) entre el psicrómetro superior (EL 1) y el inferior (EL 6) se simplifica algebraicamente:

$$\beta = \frac{-\rho c_p K_h \left(\frac{\Delta T}{\Delta z}\right)}{-\frac{\rho L_v \epsilon}{P} K_w \left(\frac{\Delta e_a}{\Delta z}\right)} = \gamma \frac{\Delta T}{\Delta z} \cdot \frac{\Delta z}{\Delta e_a} = \gamma \frac{T_1 - T_6}{e_1 - e_6}$$

<div style="background-color: #f6f8fa; border-left: 4px solid #0969da; padding: 12px 16px; border-radius: 6px; margin: 15px 0; color: #24292f;">
    <p style="color: #0969da; font-weight: bold; margin-top: 0; margin-bottom: 6px;">
        📌 Nota Teórica: Cancelación de la Altura
    </p>
    <p style="margin: 0;">
        La altura $\Delta z$ no se necesita para el cálculo directo de $\beta$ entre 2 puntos porque se cancela en el cociente. Sin embargo, sí es indispensable si se va a ajustar un perfil con 3 o más niveles usando regresión logarítmica $\ln(z)$, o si se calcula flujos por el método aerodinámico independiente. Aunque si se desea en un futuro añadir más psicrométricos (recomendable), es importante registrar la diferencia de altura entre ellas, ya que la ecuación cambia cuando se trata de 3 o más psicrométricos.
    </p>
</div>

---

##  ¿De dónde sale la Constante Psicrométrica ($\gamma$)?

La constante psicrométrica surge del **balance termodinámico de energía** en la superficie de evaporación de un psicrómetro. En el punto de equilibrio térmico, la tasa de calor sensible entregado por el aire por enfriamiento es igual a la tasa de calor latente absorbido por el agua para evaporarse ($H = LE$).

$$\rho c_p (T - T_w) = \rho \frac{\epsilon}{P} L_v (e_s(T_w) - e_a)$$

Al simplificar la densidad del aire ($\rho$) y agrupar las propiedades físicas del aire y del agua, se obtiene la expresión para $\gamma$:

$$\gamma = \frac{c_p P}{\epsilon L_v}$$

Donde:
* **$c_p$**: Calor específico del aire seco ($\approx 1.005 \times 10^{-3} \text{ MJ}/(\text{kg}\cdot^\circ\text{C})$).
* **$P$**: Presión atmosférica local en $\text{kPa}$.
* **$\epsilon$**: Relación de masas molares entre agua y aire seco ($\frac{M_w}{M_d} \approx 0.622$).
* **$L_v$**: Calor latente de vaporización del agua ($\approx 2.45 \text{ MJ/kg}$ a $20^\circ\text{C}$).

<div style="background-color: #fffbe6; border-left: 4px solid #d97706; padding: 12px 16px; border-radius: 6px; margin: 15px 0; color: #24292f;">
    <p style="color: #d97706; font-weight: bold; margin-top: 0; margin-bottom: 6px;">
        ⚠️ Advertencia sobre la Altitud y la Presión
    </p>
    <p style="margin: 0;">
        La constante $\gamma$ <b>NO es un valor universal fijo</b>. Como depende directamente de la presión atmosférica ($P$), cambia significativamente según la altitud sobre el nivel del mar:
    </p>
    <ul style="margin-top: 6px; margin-bottom: 6px; padding-left: 20px;">
        <li><b>A nivel del mar ($P \approx 101.3\text{ kPa}$):</b> $\gamma \approx 0.0668\text{ kPa/}^\circ\text{C}$</li>
        <li><b>A 2,600 m.s.n.m. (ej. Bogotá, Universidad Nacional de Colombia, $P \approx 75.4\text{ kPa}$):</b> $\gamma \approx 0.05017\text{ kPa/}^\circ\text{C}$</li>
    </ul>
    <p style="margin: 0;">
        Por ello es importante calcular $\gamma$ con la presión media del sitio de monitoreo.
    </p>
</div>

<div style="background-color: #f0fdf4; border-left: 4px solid #16a34a; padding: 12px 16px; border-radius: 6px; margin: 15px 0; color: #24292f;">
    <p style="color: #16a34a; font-weight: bold; margin-top: 0; margin-bottom: 6px;">
        💡 Dato Curioso: Interpretación Física de $\gamma$
    </p>
    <p style="margin: 0;">
        Físicamente, $\gamma$ actúa como un "factor de cambio de divisa": nos dice cuántos kilopascales ($\text{kPa}$) de presión de vapor equivalen térmicamente a $1^\circ\text{C}$ de cambio en la temperatura del aire.
    </p>
</div>

---

##  Voy a calcular el factor de Bowen con una frecuencia de 30 min (mínimo) ... Importante!!⚠️

Al utilizar **únicamente 2 niveles de medición** (EL 1 y EL 6), el cálculo queda expuesto a fluctuaciones extremas debido a:

1. **Denominador cercano a cero ($\Delta e_a \to 0$):** Si la diferencia de humedad entre niveles es diminuta, cualquier pequeña variación convierte a $\Delta e_a$ en un valor cercano a cero, disparando $\beta$ hacia $\pm \infty$.
2. **Ruido instrumental:** En un gradiente corto (ej. $1\text{ m}$), un error aleatorio de apenas $\pm 0.1^\circ\text{C}$ o $\pm 1\%$ HR representa una fracción enorme del gradiente real.
3. **Inversión de régimen en amanecer / atardecer:** Cuando la radiación neta pasa por cero, $\Delta T$ y $\Delta e_a$ no cruzan por cero en el mismo instante, generando valores negativos o divergentes.

<div style="background-color: #fef2f2; border-left: 4px solid #dc2626; padding: 12px 16px; border-radius: 6px; margin: 15px 0; color: #24292f;">
    <p style="color: #dc2626; font-weight: bold; margin-top: 0; margin-bottom: 6px;">
        🚨 Atención sobre el Promediado
    </p>
    <p style="margin: 0;">
        <b>Nunca se debe calcular $\beta$ sobre mediciones instantáneas o de alta frecuencia</b> (ej. cada 10 minutos) para luego intentar promediar las $\beta$. Las asíntotas en el denominador sesgarán por completo el promedio final.
    </p>
</div>

---

###  Protocolo de Control de Calidad (QC)

Para obtener datos científicamente válidos, se recomienda hacerlo de la siguiente forma:

1. **Promediar las variables ANTES de calcular $\beta$:** Calcular la media de $T_1, T_6, e_1, e_6$ en ventanas de **30 a 60 minutos** y luego aplica la fórmula de Bowen.
2. **Filtrar por humedad mínima:** Asignar `NaN` (Not a Number) si $|\Delta e_a| < 0.005 \text{ kPa}$.
3. **Remover rangos inestables y no físicos:**
   * Excluir la zona de falla del balance de energía: $-1.2 < \beta < -0.8$ (donde $1 + \beta \approx 0$).
   * Limitar a valores físicos plausibles: $-5 \le \beta \le 10$.

Otra opcion para validar es la agrupacion por $1/2 hr, 1 hr, 3 hr, 6 hr, 8 hr$ ... estoy viendo si lo hago

In [60]:
def unificar_psicrometros(d1, d2, col_tiempo = 'Time', tolerancia_min = 3):
    """
    Unifica dos DataFrames de psicrómetros alineando sus registros por el tiempo 
    más cercano dentro de un margen de tolerancia.

    Parámetros:
    -----------
    col_tiempo : str
        Nombre de la columna que contiene las marcas de tiempo (por defecto 'Time').
    tolerancia_min : int
        Margen máximo en minutos para emparejar dos lecturas (por defecto 3 min).

    Retorna:
    --------
    pd.DataFrame:
        DataFrame unificado con índice de tipo DatetimeIndex, ordenado cronológicamente
        y con sufijos '_z1' y '_z6' en las columnas.

    Por que es importante esta funcion? Actualmente los psicrometricos estan programados
    para guardar informacion cada 10 min sin embargo, los sensores muchas veces se 
    congelan y terminan guardando la informacion un tiempo despues de la programada.

    Por ejemplo, en lugar de guardar un dato a las 10:10:00 am, puede llegar a guardar
    el dato a las 10:10:05 am... el sensor se congelo!

    ESto genera una incompatibilidad a la hora de buscar los datos del ps1 y el ps2 en 
    la misma hora... porque no los va a encontrar ... a su vez esto genera muchos NaN 
    falsos!!
    """

    #  Garantizar conversión estricta a Datetime -> formato de tiempo correcto para pd
    d1[col_tiempo] = pd.to_datetime(d1[col_tiempo], errors = 'coerce')
    d2[col_tiempo] = pd.to_datetime(d2[col_tiempo], errors = 'coerce')

    #  Eliminar filas sin fecha y ordenar por tiempo (obligatorio para merge_asof)
    d1 = d1.dropna(subset=[col_tiempo]).sort_values(by = col_tiempo)
    d2 = d2.dropna(subset=[col_tiempo]).sort_values(by = col_tiempo)

    #  Sincronización exacta por aproximación de tiempo más cercano... la funcion real que soluciona el problema
    df_unificado = pd.merge_asof(
        d1,
        d2,
        on = col_tiempo,
        direction = 'nearest',
        tolerance = pd.Timedelta(minutes = tolerancia_min),
        suffixes = ('_z1', '_z6') #Coloca al final de cada columna el sufijo _z1 o _Z6 para diferenciarlos
    )

    #  Fijar el tiempo unificado como DatetimeIndex y ordenar
    df_unificado.set_index(col_tiempo, inplace = True)
    df_unificado.sort_index(inplace = True)

    return df_unificado

In [61]:
def calcular_beta(t1, t6, ea1, ea6, gamma) :
    """
    Calcula la Razón de Bowen (beta) a partir de gradientes verticales microclimáticos.

    Ecuación:
    ---------
    beta = gamma * ((t6 - t1) / (ea6 - ea1))

    Retorna:
    --------
    float, np.ndarray o pd.Series:
        Valor de la Razón de Bowen (beta).
    """
    
    delta_T = t6 - t1
    delta_e = ea6 - ea1
    
    # Cálculo directo
    beta = gamma * (delta_T / delta_e)

    return pd.DataFrame({'delta_ea': delta_e, 'delta_T': delta_T, 'beta': beta})

In [62]:
def obtener_gamma(altitud_m):
    P_kpa = 101.325 * (1 - 2.25577e-5 * altitud_m) ** 5.25588
    gamma = (1.005e-3 * P_kpa) / (0.622 * 2.45)
    return gamma

In [63]:
df_psicrometros = unificar_psicrometros(
    df_ps1, 
    df_ps2, 
    col_tiempo = 'Time', # Cambair en dado caso que la columna del tiempo de los psicormetricos no sea Time
    tolerancia_min = 1
)

In [64]:
df_psicrometros[['e0_kPa_z1', 'ea_kPa_z1', 'vpd_kPa_z1']] = calcular_presiones_vapor(
    df_psicrometros['Celsius(°C)_z1'],
    df_psicrometros['Humidity(%rh)_z1']/100
)
df_psicrometros[['e0_kPa_z6', 'ea_kPa_z6', 'vpd_kPa_z6']] = calcular_presiones_vapor(
    df_psicrometros['Celsius(°C)_z6'],
    df_psicrometros['Humidity(%rh)_z6']/100
)
df_psicrometros.loc['2026-07-14 07:00:00':'2026-07-15 17:00:00', ['Celsius(°C)_z1','Celsius(°C)_z6','ea_kPa_z1','ea_kPa_z6']]

,Celsius(°C)_z1,Celsius(°C)_z6,ea_kPa_z1,ea_kPa_z6
Time,,,,
2026-07-14 07:00:00,13.0,14.5,1.355483,1.263183
2026-07-14 07:10:00,13.5,14.5,1.415933,1.271439
2026-07-14 07:20:00,14.0,15.5,1.414765,1.303157
2026-07-14 07:30:00,15.0,17.5,1.492178,1.389991
2026-07-14 07:40:00,15.5,18.0,1.435234,1.413833
...,...,...,...,...
2026-07-15 16:20:00,25.0,32.5,1.726439,1.760680
2026-07-15 16:30:00,24.5,31.0,1.706430,1.684722
2026-07-15 16:40:00,23.0,26.5,1.587332,1.506006


### &#x1F527; Cambiar la frecuencia del promedio de los datos de los psicrométricos

In [65]:
# Frecuencia deseada para el análisis:
# '15min' = 15 minutos | '30min' = 30 minutos (recomendada) | '1h' = 1 hora | '3h' = 3 horas | '1d' = 1 día
frec = '30min'
gamma = obtener_gamma(z)

df_bowen = df_psicrometros.resample(frec).mean(numeric_only=True)


df_bowen["dT"] = df_bowen['Celsius(°C)_z1'] - df_bowen['Celsius(°C)_z6']
df_bowen["dea"] = df_bowen['ea_kPa_z1'] - df_bowen['ea_kPa_z6']

df_bowen["beta"] = gamma * (df_bowen["dT"] / df_bowen["dea"])
df_bowen

,EL 1,Celsius(°C)_z1,Humidity(%rh)_z1,Dew Point(°C)_z1,Serial Number_z1,EL 6,Celsius(°C)_z6,Humidity(%rh)_z6,Dew Point(°C)_z6,Serial Number_z6,e0_kPa_z1,ea_kPa_z1,vpd_kPa_z1,e0_kPa_z6,ea_kPa_z6,vpd_kPa_z6,dT,dea,beta
Time,,,,,,,,,,,,,,,,,,,
2026-04-20 00:00:00,2.0,12.333333,91.000000,10.933333,64017443.0,2.0,12.333333,92.166667,11.100000,NaN,1.433842,1.304874,0.128968,1.433842,1.321629,0.112213,0.000000,-0.016754,-0.000000
2026-04-20 00:30:00,5.0,13.000000,90.000000,11.400000,NaN,5.0,12.500000,91.666667,11.200000,NaN,1.497771,1.347994,0.149777,1.449481,1.328691,0.120790,0.500000,0.019303,1.253506
2026-04-20 01:00:00,8.0,12.166667,90.166667,10.600000,NaN,8.0,12.166667,91.333333,10.800000,NaN,1.418203,1.278564,0.139639,1.418203,1.295162,0.123041,0.000000,-0.016598,-0.000000
2026-04-20 01:30:00,11.0,12.166667,90.666667,10.666667,NaN,11.0,12.000000,91.666667,10.666667,NaN,1.418203,1.286046,0.132157,1.402564,1.285684,0.116880,0.166667,0.000362,22.260479
2026-04-20 02:00:00,14.0,11.333333,91.000000,9.933333,NaN,14.0,11.333333,92.166667,10.066667,NaN,1.342229,1.221354,0.120874,1.342229,1.237038,0.105190,0.000000,-0.015684,-0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-29 12:00:00,14475.0,30.500000,42.333333,16.066667,NaN,14475.0,35.666667,33.166667,16.700000,NaN,4.459543,1.837361,2.622183,6.016634,1.908281,4.108352,-5.166667,-0.070921,3.525447
2026-07-29 12:30:00,14478.0,34.333333,34.000000,16.200000,NaN,14478.0,42.166667,26.000000,18.633333,NaN,5.421593,1.843835,3.577758,8.307135,2.149491,6.157644,-7.833333,-0.305656,1.240196
2026-07-29 13:00:00,14481.0,34.333333,34.166667,16.266667,NaN,14481.0,42.000000,25.833333,18.400000,NaN,5.424051,1.854822,3.569229,8.213617,2.118533,6.095084,-7.666667,-0.263711,1.406873


In [66]:
df_bowen.to_excel("beta.xlsx", index = True)

# &#x1F9E0; Fundamento Teórico: Flujo de Calor en el Suelo ($G$)

---

##  Definición y Rol en el Balance de Energía

El **flujo de calor en el suelo ($G$)** representa la tasa de energía térmica transportada hacia o desde la superficie del terreno por conducción por unidad de área ($\text{W m}^{-2}$ o $\text{MJ m}^{-2} \text{d}^{-1}$).

En el modelo de balance de energía de la superficie o de un invernadero, $G$ actúa como el término de almacenamiento térmico de la superficie basimétrica:

$$R_n - G = H + LE$$

Donde $R_n - G$ es la **energía neta disponible** en la superficie para ser particionada entre el calentamiento del aire por convección ($H$, flujo de calor sensible) y la evaporación/transpiración del cultivo ($LE$, flujo de calor latente).

<div style="background-color: #f6f8fa; border-left: 4px solid #0969da; padding: 12px 16px; border-radius: 6px; margin: 15px 0; color: #24292f;">
    <p style="color: #0969da; font-weight: bold; margin-top: 0; margin-bottom: 6px;">
        📌 Nota Práctica: El Suelo como Inercia Térmica en Invernaderos
    </p>
    <p style="margin: 0;">
        En invernaderos ubicados en el trópico alto (como Bogotá), el suelo funciona como una <b>batería térmica natural</b>. Durante las horas de alta radiación solar, el piso absorbe exceso de energía, moderando el sobrecalentamiento interno. Durante la noche, el suelo devuelve ese calor acumulado hacia el ambiente interno, reduciendo el riesgo de heladas o caídas bruscas de temperatura.
    </p>
</div>

---

##  Mecanismo Físico de Transferencia (Ley de Fourier)

La transferencia de calor en el perfil del suelo ocurre predominantemente por **conducción molecular**, gobernada por la **Ley de Fourier**:

$$G = -k \frac{\partial T}{\partial z}$$

Donde:
* **$k$**: Conductividad térmica del suelo ($\text{W m}^{-1} \text{K}^{-1}$), la cual depende fuertemente del contenido de humedad volumétrica ($\theta$) y la densidad aparente del suelo.
* **$\frac{\partial T}{\partial z}$**: Gradiente vertical de temperatura en el suelo ($^\circ\text{C/m}$).
* **Signo negativo ($-$)**: Indica que el calor fluye desde zonas de mayor temperatura hacia zonas de menor temperatura (en dirección opuesta al gradiente térmico).

---

##  Convención de Signos y Dinámica Diurna

El signo matemático de $G$ determina la dirección del flujo de energía:

| Periodo | Condición Térmica | Signo de $G$ | Impacto en el Balance |
| :--- | :--- | :--- | :--- |
| **Diurno (Día)** | $T_{\text{superficie}} > T_{\text{profundo}}$ | **$G > 0$** (Positivo) | El suelo **absorbe** energía de la radiación ($R_n$). Se resta de la energía disponible para $H$ y $LE$. |
| **Nocturno (Noche)** | $T_{\text{superficie}} < T_{\text{profundo}}$ | **$G < 0$** (Negativo) | El suelo **libera** calor hacia el aire interno del invernadero. Actúa como fuente de energía activa. |

<div style="background-color: #f0fdf4; border-left: 4px solid #16a34a; padding: 12px 16px; border-radius: 6px; margin: 15px 0; color: #24292f;">
    <p style="color: #16a34a; font-weight: bold; margin-top: 0; margin-bottom: 6px;">
        💡 Dato Curioso: La Amplitud Térmica y la Humedad
    </p>
    <p style="margin: 0;">
        Un suelo húmedo conducirá calor hasta <b>4 o 5 veces más rápido</b> que un suelo seco ($k_{\text{agua}} \gg k_{\text{aire}}$). Por esta razón, el riego dentro del invernadero altera directamente el valor de $G$: un piso húmedo acumulará más calor de día y liberará más energía de noche que un piso completamente seco.
    </p>
</div>

---

##  Métodos de Estimación y Medición de $G$ (opcion para mejorar en el futuro con datos reales del suelo del invernadero)

###  Estimación Empírica por Radiación Neta (FAO-56 Sub-diaria)
En ausencia de sensores de flujo en el suelo, para intervalos sub-diarios (30–60 min), $G$ suele parametrizarse como una fracción proporcional de la radiación neta ($R_n$):

$$G_{\text{día}} = c_{\text{día}} \cdot R_n \quad (R_n > 0)$$

$$G_{\text{noche}} = c_{\text{noche}} \cdot R_n \quad (R_n \le 0)$$

* Para suelo desnudo o vegetación baja: $c_{\text{día}} \approx 0.10$ a $0.20$.
* Para invernaderos con acolchado o cobertura de suelo: la fracción puede aumentar hasta $0.20 - 0.35$ debido al calentamiento del piso cubierto. Para este caso el valor se coma como $0.15$.
* Durante la noche: $c_{\text{noche}} \approx 0.50$ (indica que cerca del 50% de las pérdidas radiativas nocturnas son compensadas por el calor emergente del suelo).

ESte método se puede consultar más a profundidad en el Capítulo 3 – Datos meteorológicos *"Flujo del calor del suelo (G)"*, ecuaciones (45) y (46).

<div style="background-color: #fffbe6; border-left: 4px solid #d97706; padding: 12px 16px; border-radius: 6px; margin: 15px 0; color: #24292f;">
    <p style="color: #d97706; font-weight: bold; margin-top: 0; margin-bottom: 6px;">
        ⚠️ Advertencia sobre la Escala Temporal
    </p>
    <p style="margin: 0;">
        A escala <b>diaria completa (24 horas)</b>, $G_{\text{24h}} \approx 0$ porque la energía absorbida de día se cancela con la liberada de noche. Sin embargo, para el modelo <b>sub-diario actual (30 - 60 minutos)</b>, $G$ <b>NUNCA debe asumir cero</b>, ya que puede llegar a representar entre el $10\%$ y el $30\%$ de $R_n$ en horas pico.
    </p>
</div>

In [67]:
def calcular_G_subdiario(Rn, c_day = 0.15, c_night = 0.50):
    """Calcula el Flujo de Calor en el Suelo (G) a escala sub-diaria (10, 15,

    30 o 60 min) como fracción de la Radiación Neta (Rn) según la FAO-56 y
    literatura de invernaderos.

    Parámetros:
    -----------
    Rn : pd.Series o np.ndarray
        Serie o arreglo de Radiación Neta (Rn) en W/m² o MJ/m².
    c_day : float, opcional (por defecto 0.15)
        Fracción de Rn transferida al suelo durante el día (Rn >= 0).
        - 0.10: Cobertura vegetal densa / FAO-56 estándar.
        - 0.15: Cobertura vegetal media (Invernadero típico).
        - 0.20 - 0.25: Suelo desnudo o fase inicial del cultivo.
    c_night : float, opcional (por defecto 0.50)
        Fracción de Rn liberada por el suelo durante la noche (Rn < 0).

    Retorna:
    --------
    G : pd.Series o np.ndarray
        Flujo de calor en el suelo (G) en la misma unidad que Rn.
    """
    # Cálculo vectorial: c_day para día (Rn >= 0) y c_night para noche (Rn < 0)
    G_values = np.where(Rn >= 0, c_day * Rn, c_night * Rn)

    # Conservar la estructura de Pandas Series si la entrada es una Serie
    if isinstance(Rn, pd.Series):
        return pd.Series(G_values, index = Rn.index, name = "G")

    return G_values

In [68]:
c_day = 0.15

#  Lod daos de Rn ya estan calculados en df_est, cuyo procesamiento esta mas arriba.
#  Recordando ... esos datos vienen cada 5 min y bowen los tenemos cada 30 min... o lo que haya en la variable frec
#  Asi que hay que reordenar los datos de la estacion para tener la misma frecuencia

df_est['Rn_W_m2'] = pd.to_numeric(df_est['Rn_W_m2'], errors='coerce') # Convierte a numeros para evitar errores
df_est_frec = df_est.resample(frec).mean(numeric_only=True)

df_est_frec['G_W_m2'] = calcular_G_subdiario(
    df_est_frec['Rn_W_m2'], 
    c_day
)

df_est_frec

,e0_kPa,ea_kPa,vpd_kPa,Ra_MJ_m2,Ra_W_m2,Rso_MJ_m2,Rso_W_m2,Rnl_W_m2,Rnl_MJ_m2,Rn_W_m2,G_W_m2
z6-13472,,,,,,,,,,,
2026-04-20 00:00:00,1.433107,1.332317,0.100790,0.0,0.0,0.0,0.0,44.275020,0.013283,-44.275020,-22.137510
2026-04-20 00:30:00,1.467039,1.375030,0.092009,0.0,0.0,0.0,0.0,43.855101,0.013157,-43.855101,-21.927551
2026-04-20 01:00:00,1.425583,1.320972,0.104612,0.0,0.0,0.0,0.0,44.395799,0.013319,-44.395799,-22.197900
2026-04-20 01:30:00,1.406305,1.314791,0.091514,0.0,0.0,0.0,0.0,44.360691,0.013308,-44.360691,-22.180346
2026-04-20 02:00:00,1.364851,1.267736,0.097115,0.0,0.0,0.0,0.0,44.791034,0.013437,-44.791034,-22.395517
...,...,...,...,...,...,...,...,...,...,...,...
2026-07-14 21:30:00,1.614005,1.265493,0.348511,0.0,0.0,0.0,0.0,3.878301,0.001163,-3.878301,-1.939151
2026-07-14 22:00:00,1.510390,1.252954,0.257436,0.0,0.0,0.0,0.0,3.839966,0.001152,-3.839966,-1.919983
2026-07-14 22:30:00,1.489307,1.290809,0.198497,0.0,0.0,0.0,0.0,3.779401,0.001134,-3.779401,-1.889700


$$ETa = \frac{R_n - G}{(1 + \beta) \cdot \lambda}$$

In [69]:
lambdaa = 2.45e6  # Python reconoce lambda como una variable preestablecida, asi que la uso con doble a.

dt_seg = dt_seg = pd.Timedelta(frec).total_seconds() # 1800 segundos, actualmente frec = 30 min 

energy_disp = df_est_frec['Rn_W_m2'] - df_est_frec['G_W_m2']

#  ETa subdiaria en mm por intervalo (W/m2 * s / (J/kg) = mm)
ETa = (energy_disp / ((1 + df_bowen['beta']) * lambdaa)) * dt_seg

#  Recortar condensación nocturna negativa para acotar unicamente a la evapotranspiración efectiva
ETa = ETa.clip(lower = 0)

#  Agregación diaria (Suma de los mm de cada intervalo del día)
ETa_d = ETa.resample('1D').sum()

ETa_d

z6-13472
2026-04-20    2.322012
2026-04-21    1.491095
2026-04-22    1.883743
2026-04-23    1.073741
2026-04-24    1.626928
                ...   
2026-07-25    0.000000
2026-07-26    0.000000
2026-07-27    0.000000
2026-07-28    0.000000
2026-07-29    0.000000
Freq: D, Length: 101, dtype: float64

In [70]:
ETa_d.to_excel(f"ETa_{frec}.xlsx", index=True)

# Análisis Comparativo: $ET_a$ en Invernadero vs. Sistema Abierto

---

<div style="background-color: #f0fdf4; border-left: 4px solid #16a34a; padding: 14px 18px; border-radius: 6px; margin: 15px 0; color: #24292f;">
    <p style="color: #16a34a; font-weight: bold; font-size: 1.05em; margin-top: 0; margin-bottom: 6px;">
        💡 Conclusión Principal: ¿$ET_a$ en Invernadero es Mayor o Menor?
    </p>
    <p style="margin: 0;">
        En términos generales, la <b>Evapotranspiración Real ($ET_a$) acumulada diariamente dentro de un invernadero es MENOR</b> que la de un cultivo idéntico en campo abierto (típicamente entre un <b>20% y 40% menor</b>). 
        <br><br>
        Aunque la temperatura interna del invernadero suele ser más alta, la drástica reducción del <b>viento</b> y la atenuación de la <b>radiación solar</b> por la cubierta de polietileno/vidrio reducen significativamente la demanda atmosférica total.
    </p>
</div>

---

## 1. Cuadro Comparativo del Microclima y su Efecto en $ET_a$

| Variable Climática | Invernadero vs. Campo Abierto | Impacto Físico en la $ET_a$ |
| :--- | :--- | :--- |
| **Radiación Solar ($R_s$, $R_n$)** | **Menor ($\downarrow$ 15% - 30%)** | La cubierta atenúa la radiación entrante. Menos energía radiativa disponible para convertir agua líquida en vapor ($LE$). |
| **Velocidad del Viento ($u_2$)** | **Mucho menor ($\downarrow\downarrow$ 70% - 90%)** | La estructura atenúa el movimiento del aire ($u \approx 0.1 - 0.5 \text{ m/s}$). Aumenta la resistencia aerodinámica ($r_a$), haciendo más lento el transporte de vapor. |
| **Humedad Relativa ($HR$)** | **Mayor ($\uparrow$)** | El ambiente cerrado acumula el vapor transpirado por las plantas, aumentando la humedad interna. |
| **Deficit de Presión de Vapor ($VPD$)** | **Generalmente Menor ($\downarrow$)** | Salvo con ventilación extrema, el alto $HR$ interno reduce la diferencia de presión de vapor entre la hoja y el aire, bajando la "fuerza impulsora" de la transpiración. |
| **Temperatura ($T$)** | **Mayor ($\uparrow$)** | Aumenta la presión de vapor de saturación ($e_s$), pero no compensa la caída de radiación y viento en la suma diaria. |

---

## 2. Factores Físicos Determinantes

### A. Dominio Aerodinámico (Resistencia del Canopeo)
En la ecuación de Penman-Monteith, la evapotranspiración depende de la combinación de la **energía neta ($R_n - G$)** y el **término aerodinámico**:

$$ET_a \propto \frac{\Delta (R_n - G) + \rho_a c_p \frac{VPD}{r_a}}{\Delta + \gamma \left(1 + \frac{r_s}{r_a}\right)}$$

* En **campo abierto**, el viento continuo mantiene $r_a$ (resistencia aerodinámica) en valores muy bajos, permitiendo que la atmósfera "relojee" y remueva rápidamente el vapor de agua de la capa límite de la hoja.
* En **invernadero**, la calma chicha ($u_2 \to 0$) dispara la resistencia aerodinámica ($r_a \uparrow\uparrow$), desacoplando parcialmente el cultivo de la atmósfera externa (*decoupling factor* $\Omega \to 1$). El flujo de evapotranspiración pasa a estar estrictamente limitado por la **radiación neta disponible ($R_n$)**.

---

### B. Transmisión del Material de Cubierta
Los plásticos agrícolas (como el polietileno térmico de baja densidad usado en la sabana de Bogotá) tienen una transmitancia radiativa de $0.70 - 0.85$. Esto significa que la energía solar que llega al cultivo interno siempre será menor que la que recibiría en el exterior.

---

## 3. Consideraciones Específicas para Bogotá (Trópico Alto)

<div style="background-color: #f6f8fa; border-left: 4px solid #0969da; padding: 12px 16px; border-radius: 6px; margin: 15px 0; color: #24292f;">
    <p style="color: #0969da; font-weight: bold; margin-top: 0; margin-bottom: 6px;">
        📌 Excepción Regional: Días Fríos / Nublados o Horas de Madrugada
    </p>
    <p style="margin: 0;">
        En la Sabana de Bogotá ($2600 \text{ msnm}$, $P \approx 74 \text{ kPa}$), durante mañanas muy frías o madrugadas donde en el exterior hay inversión térmica o heladas, el interior del invernadero retiene calor sensible en el suelo ($G < 0$).
        <br><br>
        En esos momentos puntuales, la $ET_a$ subdiaria en el invernadero puede ser ligeramente <b>mayor</b> que en el campo abierto (donde las estomas del cultivo exterior se cierran por frío extremo o condensación). Sin embargo, en el <b>total diario ($\text{mm/día}$)</b>, la $ET_a$ del invernadero sigue siendo menor.
    </p>
</div>

---
